# 2026 FIFA Men's World Cup Exploratory Data Analysis

This notebook explores historical World Cup results, team participation, confederation representation, scoring patterns, winner follow-up performance, and correlation signals connected to tournament outcomes. ahead of the 2026 edition of the tournament.

## Setup

Import the data analysis, visualization, clustering, path, widget, and notebook display libraries used throughout the analysis.

In [118]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import os
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import ipywidgets as widgets
from IPython.display import clear_output, display


## Dataset Sources

The analysis combines processed World Cup files built from Kaggle datasets, GitHub data, Elo ratings, and FIFA sources. These inputs provide match results, tournament history, placements, teams, confederations, Elo movement, and scoring data.

- Historical International Results: `https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017`
- National Squads: `https://www.kaggle.com/datasets/mateusdesousamartins/world-cup-2022-national-teams-data-set`
- World Cup history: `https://www.kaggle.com/datasets/mafaqbhatti/fifa-world-cup-complete-history-19302022`
- Comprehensive Historic World Cup Database: `https://github.com/jfjelstul/worldcup`
- Elo Ratings: `eloratings.com`
- FIFA Ratings: `fifa.com`

### Load Processed Data

Load the four core processed CSV files: match results, tournament history, placement records, and team participation records.

In [119]:
results = pd.read_csv("data/processed/world_cup/all_editions/results.csv")
history = pd.read_csv("data/processed/world_cup/fifa_world_cup_history.csv")
placement = pd.read_csv("data/processed/world_cup/all_editions/placement.csv")
teams = pd.read_csv("data/processed/world_cup/all_editions/teams.csv")
pd.set_option("display.max_columns", None)

### Normalize Core Fields

Standardize shared column names, attach tournament-size context to placement records, and assign each World Cup edition to a historical era. The shared `datasets` dictionary supports the quality checks that follow.

In [120]:
#Add next edition placement for all countries
history = history.rename(columns={"Year": "edition", "Teams": "teams", "Host_Country": "host"})
teams = teams.rename(columns={"year": "edition", "team": "country"})
results = results.rename(columns={"country": "host", "team": "country"})

# Add tournament-level context. If placement.csv already has a host column,
# keep it and only fill missing host values from the history table.
placement_context = history[["edition", "teams", "host"]].drop_duplicates("edition")
placement = placement.merge(
   placement_context,
    on="edition",
    how="left",
    suffixes=("", "_history"),
    validate="many_to_one"
    )

for column in ["teams", "host"]:
    history_column = f"{column}_history"
    if history_column in placement.columns:
        placement[column] = placement[column].fillna(placement[history_column])
        placement = placement.drop(columns=history_column)

#Add eras
era_bins = [1929, 1950, 1970, 1990, 2010, 2026]
era_labels = [
    "Early Era",
    "Golden Age",
    "Modern Era",
    "Contemporary",
    "Recent",
]

teams["era"] = pd.cut(
    teams["edition"],
    bins=era_bins,
    labels=era_labels,
)

results["era"] = pd.cut(
    results["edition"],
    bins=era_bins,
    labels=era_labels,
)

#Add ELO rankings for evry nation
placement["elo_rank"] = (
    placement.groupby(["edition", "tournament_id"])["start_elo"]
    .rank(method="min", ascending=False)
    .astype(int)
)


#fix empty next edition, placement and position
print(placement["next_placement"].isnull().sum())

all_editions = sorted(set(placement["edition"].dropna()) | set(teams["edition"].dropna()))
next_edition_map = dict(zip(all_editions[:-1], all_editions[1:]))
latest_completed_edition = placement["edition"].max()

placement["expected_next_edition"] = placement["edition"].map(next_edition_map)
placement["next_edition"] = placement["next_edition"].fillna(placement["expected_next_edition"])

team_next_lookup = teams[["edition", "country"]].drop_duplicates()
team_next_lookup["qualified_next_edition"] = True

placement = placement.merge(
    team_next_lookup,
    left_on=["expected_next_edition", "country"],
    right_on=["edition", "country"],
    how="left",
    suffixes=("", "_next_lookup"),
)

placement["qualified_next_edition"] = placement["qualified_next_edition"].fillna(False)

mask_has_next_edition = placement["expected_next_edition"].notna()
mask_missing_next_placement = placement["next_placement"].isna()
mask_future_next_results = placement["expected_next_edition"].gt(latest_completed_edition)

placement.loc[
    mask_has_next_edition & mask_missing_next_placement & ~placement["qualified_next_edition"],
    "next_placement",
] = "DNQ"

placement.loc[
    mask_has_next_edition
    & mask_missing_next_placement
    & placement["qualified_next_edition"]
    & mask_future_next_results,
    "next_placement",
] = "TBD"

placement["next_edition"] = pd.to_numeric(placement["next_edition"], errors="coerce").astype("Int64").astype("object")
placement = placement.drop(columns=["expected_next_edition", "edition_next_lookup", "qualified_next_edition"])
#placement["next_position"] = placement["next_position"].fillna(0)
display(placement.head(32))


#print(f"Results CSV colums: {results.columns}")
#print(f"History CSV colums: {history.columns}")
print(f"Placement CSV colums: {placement.columns}")
#print(f"Teams CSV colums: {teams.columns}")

datasets = {
    "results"  : results,
    "history"  : history,
    "placement": placement,
    "teams"    : teams,
}

#display(placement)


84


C:\Users\oukan\AppData\Local\Temp\ipykernel_9736\1007054215.py:74: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  placement["qualified_next_edition"] = placement["qualified_next_edition"].fillna(False)


,edition,tournament_id,year,country,team_id,team_code,confederation,placement,position,matches_played,gs,ga,start_elo,finish_elo,elo_change,next_edition,next_placement,next_position,teams,host,elo_rank
0,1930,WC-1930,1930,Uruguay,URY,URY,CONMEBOL,Winner,1,4,15,3,1968,2034,66,1950,Winner,1.0,13,Uruguay,2
1,1930,WC-1930,1930,Argentina,ARG,ARG,CONMEBOL,Runner-up,2,5,18,9,2073,2059,-14,1934,Round of 16,16.0,13,Uruguay,1
2,1930,WC-1930,1930,United States,USA,USA,CONCACAF,Third Place,3,3,7,6,1672,1766,94,1934,Round of 16,16.0,13,Uruguay,5
3,1930,WC-1930,1930,Yugoslavia,YUG,YUG,UEFA,Fourth Place,4,3,7,7,1590,1643,53,1950,Group Stage,13.0,13,Uruguay,8
4,1930,WC-1930,1930,Belgium,BEL,BEL,UEFA,Group Stage,13,2,0,4,1661,1589,-72,1934,Round of 16,16.0,13,Uruguay,6
5,1930,WC-1930,1930,Bolivia,BOL,BOL,CONMEBOL,Group Stage,13,2,0,8,1258,1244,-14,1950,Group Stage,13.0,13,Uruguay,13
6,1930,WC-1930,1930,Brazil,BRA,BRA,CONMEBOL,Group Stage,13,2,5,2,1923,1874,-49,1934,Round of 16,16.0,13,Uruguay,3
7,1930,WC-1930,1930,Chile,CHL,CHL,CONMEBOL,Group Stage,13,3,5,3,1575,1645,70,1950,Group Stage,13.0,13,Uruguay,9
8,1930,WC-1930,1930,France,FRA,FRA,UEFA,Group Stage,13,3,4,3,1547,1577,30,1934,Round of 16,16.0,13,Uruguay,11
9,1930,WC-1930,1930,Mexico,MEX,MEX,CONCACAF,Group Stage,13,3,4,13,1613,1498,-115,1950,Group Stage,13.0,13,Uruguay,7


Placement CSV colums: Index(['edition', 'tournament_id', 'year', 'country', 'team_id', 'team_code',
       'confederation', 'placement', 'position', 'matches_played', 'gs', 'ga',
       'start_elo', 'finish_elo', 'elo_change', 'next_edition',
       'next_placement', 'next_position', 'teams', 'host', 'elo_rank'],
      dtype='object')


### Missing Elo Review

Identify teams with missing Elo movement in the match-results file before using Elo-related fields in later analysis.

In [121]:
#display(results.loc[results["team_elo_delta"].isnull()])
missing_elo = results.loc[results["team_elo_delta"].isnull()]

print(missing_elo['country'].unique())

[]


### Data Quality Snapshot

Print each dataset's shape, size, duplicate count, data types, missing-value percentages, missing-value counts, and rows containing nulls. This provides a quick completeness check before the exploratory sections.

In [122]:
# GLOBAL HEALTH CHECK 
for name, df in datasets.items():
    print(f"\n{'='*55}")
    print(f"  {name.upper()}")
    print(f"{'='*55}")
    print(f"  Shape      : {df.shape}")
    print(f"  Size       : {df.size:,} cells")
    print(f"  Duplicates : {df.duplicated().sum()}")
    print(f"\n  Dtypes:")
    print(df.dtypes.to_string())
    print(f"\n  Missing Values (%):")
    missing = (df.isnull().mean() * 100).sort_values(ascending=False)
    print(missing[missing > 0].to_string() if missing.any() else "  None")
    print(f"\n  Missing Values:")
    missing = (df.isnull().sum()).sort_values(ascending=False)
    print(missing[missing > 0].to_string() if missing.any() else "  None")

    display(df.loc[df.isnull().any(axis=1), :])


  RESULTS
  Shape      : (1928, 28)
  Size       : 53,984 cells
  Duplicates : 0

  Dtypes:
edition                      int64
tournament_id               object
match_id                    object
match_number                 int64
date                        object
stage                       object
status                      object
team_id                     object
country                     object
team_confederation          object
opponent_id                 object
opponent                    object
opponent_confederation      object
is_home                       bool
team_score                   int64
opponent_score               int64
result                      object
team_elo_start               int64
opponent_elo_start           int64
team_elo_end                 int64
opponent_elo_end             int64
team_elo_delta               int64
city                        object
host                        object
neutral                       bool
decided_by_shootout           bo

,edition,tournament_id,match_id,match_number,date,stage,status,team_id,country,team_confederation,opponent_id,opponent,opponent_confederation,is_home,team_score,opponent_score,result,team_elo_start,opponent_elo_start,team_elo_end,opponent_elo_end,team_elo_delta,city,host,neutral,decided_by_shootout,shootout_winner,era
0,1930,WC-1930,WC-1930_001,1,1930-07-13,Group Stage,played,BEL,Belgium,UEFA,USA,United States,CONCACAF,True,0,3,loss,1661,1672,1610,1723,-51,Montevideo,Uruguay,True,False,NaN,Early Era
1,1930,WC-1930,WC-1930_001,1,1930-07-13,Group Stage,played,USA,United States,CONCACAF,BEL,Belgium,UEFA,False,3,0,win,1672,1661,1723,1610,51,Montevideo,Uruguay,True,False,NaN,Early Era
2,1930,WC-1930,WC-1930_002,2,1930-07-13,Group Stage,played,FRA,France,UEFA,MEX,Mexico,CONCACAF,True,4,1,win,1547,1613,1609,1551,62,Montevideo,Uruguay,True,False,NaN,Early Era
3,1930,WC-1930,WC-1930_002,2,1930-07-13,Group Stage,played,MEX,Mexico,CONCACAF,FRA,France,UEFA,False,1,4,loss,1613,1547,1551,1609,-62,Montevideo,Uruguay,True,False,NaN,Early Era
4,1930,WC-1930,WC-1930_003,3,1930-07-14,Group Stage,played,BRA,Brazil,CONMEBOL,YUG,Yugoslavia,UEFA,True,1,2,loss,1923,1590,1871,1642,-52,Montevideo,Uruguay,True,False,NaN,Early Era
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1921,2022,WC-2022,WC-2022_061,61,2022-12-13,Semi-final,played,HRV,Croatia,UEFA,ARG,Argentina,CONMEBOL,False,0,3,loss,1952,2121,1923,2150,-29,Lusail,Qatar,True,False,NaN,Recent
1922,2022,WC-2022,WC-2022_062,62,2022-12-14,Semi-final,played,FRA,France,UEFA,MAR,Morocco,CAF,True,2,0,win,2046,1923,2076,1893,30,Al Khor,Qatar,True,False,NaN,Recent
1923,2022,WC-2022,WC-2022_062,62,2022-12-14,Semi-final,played,MAR,Morocco,CAF,FRA,France,UEFA,False,0,2,loss,1923,2046,1893,2076,-30,Al Khor,Qatar,True,False,NaN,Recent
1924,2022,WC-2022,WC-2022_063,63,2022-12-17,Third Place,played,HRV,Croatia,UEFA,MAR,Morocco,CAF,True,2,1,win,1923,1893,1950,1866,27,Al Rayyan,Qatar,True,False,NaN,Recent



  HISTORY
  Shape      : (22, 10)
  Size       : 220 cells
  Duplicates : 0

  Dtypes:
edition              int64
host                object
Winner              object
Runner_Up           object
Third_Place         object
Fourth_Place        object
Total_Goals          int64
Matches_Played       int64
teams                int64
Goals_Per_Match    float64

  Missing Values (%):
  None

  Missing Values:
  None


,edition,host,Winner,Runner_Up,Third_Place,Fourth_Place,Total_Goals,Matches_Played,teams,Goals_Per_Match



  PLACEMENT
  Shape      : (489, 21)
  Size       : 10,269 cells
  Duplicates : 0

  Dtypes:
edition             int64
tournament_id      object
year                int64
country            object
team_id            object
team_code          object
confederation      object
placement          object
position            int64
matches_played      int64
gs                  int64
ga                  int64
start_elo           int64
finish_elo          int64
elo_change          int64
next_edition       object
next_placement     object
next_position     float64
teams               int64
host               object
elo_rank            int64

  Missing Values (%):
next_position    17.177914

  Missing Values:
next_position    84


,edition,tournament_id,year,country,team_id,team_code,confederation,placement,position,matches_played,gs,ga,start_elo,finish_elo,elo_change,next_edition,next_placement,next_position,teams,host,elo_rank
33,1938,WC-1938,1938,Cuba,CUB,CUB,CONCACAF,Quarter-final,8,3,5,12,1543,1518,-25,1950,DNQ,NaN,15,France,14
39,1938,WC-1938,1938,Indonesia,IDN,IDN,AFC,Round of 16,16,1,0,6,1468,1460,-8,1950,DNQ,NaN,15,France,15
133,1970,WC-1970,1970,Israel,ISR,ISR,UEFA,Group Stage,16,3,1,3,1682,1696,14,1974,DNQ,NaN,16,Mexico,14
141,1974,WC-1974,1974,German DR,DDR,DDR,UEFA,Quarter-final,8,6,5,5,1939,1918,-21,1978,DNQ,NaN,16,West Germany,5
148,1974,WC-1974,1974,DR Congo,COD,COD,CAF,Group Stage,16,3,0,14,1780,1695,-85,1978,DNQ,NaN,16,West Germany,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484,2022,WC-2022,2022,Saudi Arabia,KSA,SAU,AFC,Group Stage,32,3,3,5,1636,1644,8,2026,TBD,NaN,32,Qatar,30
485,2022,WC-2022,2022,Serbia,SRB,SRB,UEFA,Group Stage,32,3,5,8,1898,1835,-63,2026,DNQ,NaN,32,Qatar,14
486,2022,WC-2022,2022,Tunisia,TUN,TUN,CAF,Group Stage,32,3,1,1,1705,1745,40,2026,TBD,NaN,32,Qatar,27
487,2022,WC-2022,2022,Uruguay,URY,URY,CONMEBOL,Group Stage,32,3,2,2,1935,1904,-31,2026,TBD,NaN,32,Qatar,10



  TEAMS
  Shape      : (537, 11)
  Size       : 5,907 cells
  Duplicates : 0

  Dtypes:
tournament_id        object
edition               int64
team_id              object
country              object
team_code            object
confederation        object
tournament_name      object
placement            object
position            float64
matches_played      float64
era                category

  Missing Values (%):
matches_played    8.938547
position          8.938547

  Missing Values:
matches_played    48
position          48


,tournament_id,edition,team_id,country,team_code,confederation,tournament_name,placement,position,matches_played,era
489,WC-2026,2026,CZE,Czechia,CZE,UEFA,2026 FIFA Men's World Cup,Scheduled,NaN,NaN,Recent
490,WC-2026,2026,KOR,South Korea,KOR,AFC,2026 FIFA Men's World Cup,Scheduled,NaN,NaN,Recent
491,WC-2026,2026,MEX,Mexico,MEX,CONCACAF,2026 FIFA Men's World Cup,Scheduled,NaN,NaN,Recent
492,WC-2026,2026,RSA,South Africa,RSA,CAF,2026 FIFA Men's World Cup,Scheduled,NaN,NaN,Recent
493,WC-2026,2026,BIH,Bosnia and Herzegovina,BIH,UEFA,2026 FIFA Men's World Cup,Scheduled,NaN,NaN,Recent
494,WC-2026,2026,CAN,Canada,CAN,CONCACAF,2026 FIFA Men's World Cup,Scheduled,NaN,NaN,Recent
495,WC-2026,2026,QAT,Qatar,QAT,AFC,2026 FIFA Men's World Cup,Scheduled,NaN,NaN,Recent
496,WC-2026,2026,SUI,Switzerland,SUI,UEFA,2026 FIFA Men's World Cup,Scheduled,NaN,NaN,Recent
497,WC-2026,2026,BRA,Brazil,BRA,CONMEBOL,2026 FIFA Men's World Cup,Scheduled,NaN,NaN,Recent
498,WC-2026,2026,HAI,Haiti,HAI,CONCACAF,2026 FIFA Men's World Cup,Scheduled,NaN,NaN,Recent


## Match Results EDA

Summarize the match-level dataset, including date coverage, editions, stages, confederation counts, match outcomes, shootouts, score distributions, Elo distributions, and key null fields.

In [123]:
# ============================================================
# 3. RESULTS — Full EDA (Primary Dataset)
# ============================================================

display(results.sample(5))
display(results.describe(include='all').T)

# Date parsing
results['date'] = pd.to_datetime(results['date'])
display(f"\nDate range: {results['date'].min()} → {results['date'].max()}")

# Editions covered
display(f"Editions   : {sorted(results['edition'].unique())}")
display(f"Stages     : {results['stage'].unique()}")

# Confederation distribution
display("\nTeam Confederation counts:")
display(results['team_confederation'].value_counts())

# Result distribution (W/L/D)
display("\nMatch Results:")
display(results['result'].value_counts())

# Shootout matches
display(f"\nDecided by shootout: {results['decided_by_shootout'].sum()}")

# Score stats
display("\nScore descriptive stats:")
display(results[['team_score', 'opponent_score']].describe())

# ELO stats
display("\nELO descriptive stats:")
display(results[['team_elo_start', 'opponent_elo_start', 
               'team_elo_delta']].describe())

# Nulls to watch — shootout_winner will be null for non-shootout games
display("\nNulls in shootout_winner:", results['shootout_winner'].isnull().sum())
display("(Expected: all non-shootout matches)")

,edition,tournament_id,match_id,match_number,date,stage,status,team_id,country,team_confederation,opponent_id,opponent,opponent_confederation,is_home,team_score,opponent_score,result,team_elo_start,opponent_elo_start,team_elo_end,opponent_elo_end,team_elo_delta,city,host,neutral,decided_by_shootout,shootout_winner,era
600,1978,WC-1978,WC-1978_031,31,1978-06-18,Quarter-final,played,GER,Germany,UEFA,NED,Netherlands,UEFA,True,2,2,draw,2071,2077,2072,2076,1,Córdoba,Argentina,True,False,NaN,Modern Era
27,1930,WC-1930,WC-1930_014,14,1930-07-21,Group Stage,played,ROU,Romania,UEFA,URY,Uruguay,CONMEBOL,False,0,4,loss,1603,1970,1596,1977,-7,Montevideo,Uruguay,False,False,NaN,Early Era
1681,2018,WC-2018,WC-2018_005,5,2018-06-16,Group Stage,played,AUS,Australia,AFC,FRA,France,UEFA,False,1,2,loss,1743,1986,1731,1998,-12,Kazan,Russia,True,False,NaN,Recent
435,1970,WC-1970,WC-1970_018,18,1970-06-10,Group Stage,played,SUN,Soviet Union,UEFA,SLV,El Salvador,CONCACAF,False,2,0,win,1960,1496,1966,1490,6,Mexico City,Mexico,True,False,NaN,Golden Age
1629,2014,WC-2014,WC-2014_043,43,2014-06-25,Group Stage,played,CHE,Switzerland,UEFA,HND,Honduras,CONCACAF,False,3,0,win,1835,1630,1860,1605,25,Manaus,Brazil,True,False,NaN,Recent


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
edition,1928.0,NaN,NaN,NaN,1989.244813,24.008177,1930.0,1974.0,1994.0,2010.0,2022.0
tournament_id,1928,22,WC-2006,128,NaN,NaN,NaN,NaN,NaN,NaN,NaN
match_id,1928,964,WC-1930_001,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
match_number,1928.0,NaN,NaN,NaN,25.795643,17.071745,1.0,11.0,23.0,38.0,64.0
date,1928,378,1958-06-15,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
stage,1928,6,Group Stage,1352,NaN,NaN,NaN,NaN,NaN,NaN,NaN
status,1928,1,played,1928,NaN,NaN,NaN,NaN,NaN,NaN,NaN
team_id,1928,84,BRA,114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,1928,84,Brazil,114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
team_confederation,1928,6,UEFA,1083,NaN,NaN,NaN,NaN,NaN,NaN,NaN


'\nDate range: 1930-07-13 00:00:00 → 2022-12-18 00:00:00'

'Editions   : [np.int64(1930), np.int64(1934), np.int64(1938), np.int64(1950), np.int64(1954), np.int64(1958), np.int64(1962), np.int64(1966), np.int64(1970), np.int64(1974), np.int64(1978), np.int64(1982), np.int64(1986), np.int64(1990), np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014), np.int64(2018), np.int64(2022)]'

"Stages     : ['Group Stage' 'Semi-final' 'Final' 'Round of 16' 'Quarter-final'\n 'Third Place']"

...

'\nELO descriptive stats:'

,team_elo_start,opponent_elo_start,team_elo_delta
count,1928.000000,1928.000000,1928.000000
mean,1865.259336,1865.259336,0.000000
std,147.311869,147.311869,29.712583
min,1247.000000,1247.000000,-83.000000
25%,1762.000000,1762.000000,-21.000000
50%,1880.000000,1880.000000,0.000000
75%,1976.000000,1976.000000,21.000000
max,2234.000000,2234.000000,83.000000


'\nNulls in shootout_winner:'

np.int64(1858)

'(Expected: all non-shootout matches)'

## Placement EDA

Inspect tournament placement records by edition, confederation, finishing position, Elo change, goals for and against, matches played, and next-edition fields.

In [124]:
# ============================================================
# 4. PLACEMENT — Full EDA (Primary Dataset)
# ============================================================

display(placement.sample(5))
display(placement.describe(include='all').T)

# Editions & years
display(f"\nEditions  : {sorted(placement['edition'].unique())}")
display(f"Year range: {placement['year'].min()} → {placement['year'].max()}")

# Confederation breakdown
display("\nConfederation counts:")
display(placement['confederation'].value_counts())

# Placement / position distribution
display("\nPlacement value counts:")
display(placement['placement'].value_counts().head(10))

# ELO change overview
display("\nELO Change stats:")
display(placement[['start_elo', 'finish_elo', 'elo_change']].describe())

# Goals for/against
display("\nGoals stats:")
display(placement[['gs', 'ga', 'matches_played']].describe())

# next_edition / next_placement — will have nulls for the latest edition
display(f"\nNulls in next_edition   : {placement['next_edition'].isnull().sum()}")
display(f"Nulls in next_placement : {placement['next_placement'].isnull().sum()}")

,edition,tournament_id,year,country,team_id,team_code,confederation,placement,position,matches_played,gs,ga,start_elo,finish_elo,elo_change,next_edition,next_placement,next_position,teams,host,elo_rank
455,2018,WC-2018,2018,South Korea,KOR,KOR,AFC,Group Stage,32,3,3,3,1713,1756,43,2022,Round of 16,16.0,32,Russia,25
52,1950,WC-1950,1950,Mexico,MEX,MEX,CONCACAF,Group Stage,13,3,2,10,1834,1729,-105,1954,Group Stage,16.0,13,Brazil,7
145,1974,WC-1974,1974,Australia,AUS,AUS,AFC,Group Stage,16,3,0,5,1687,1676,-11,2006,Round of 16,16.0,16,West Germany,15
445,2018,WC-2018,2018,Iceland,ISL,ISL,UEFA,Group Stage,32,3,2,5,1764,1708,-56,2022,DNQ,NaN,32,Russia,20
426,2018,WC-2018,2018,Croatia,HRV,HRV,UEFA,Runner-up,2,7,14,9,1853,1942,89,2022,Third Place,3.0,32,Russia,14


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
edition,489.0,NaN,NaN,NaN,1987.186094,26.098178,1930.0,1970.0,1994.0,2010.0,2022.0
tournament_id,489,22,WC-2006,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,489.0,NaN,NaN,NaN,1987.186094,26.098178,1930.0,1970.0,1994.0,2010.0,2022.0
country,489,84,Brazil,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN
team_id,489,84,BRA,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN
team_code,489,84,BRA,22,NaN,NaN,NaN,NaN,NaN,NaN,NaN
confederation,489,6,UEFA,259,NaN,NaN,NaN,NaN,NaN,NaN,NaN
placement,489,8,Group Stage,222,NaN,NaN,NaN,NaN,NaN,NaN,NaN
position,489.0,NaN,NaN,NaN,16.380368,10.33132,1.0,8.0,16.0,24.0,32.0
matches_played,489.0,NaN,NaN,NaN,3.94274,1.46852,1.0,3.0,3.0,5.0,7.0


'\nEditions  : [np.int64(1930), np.int64(1934), np.int64(1938), np.int64(1950), np.int64(1954), np.int64(1958), np.int64(1962), np.int64(1966), np.int64(1970), np.int64(1974), np.int64(1978), np.int64(1982), np.int64(1986), np.int64(1990), np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014), np.int64(2018), np.int64(2022)]'

'Year range: 1930 → 2022'

'\nConfederation counts:'

...

,start_elo,finish_elo,elo_change
count,489.000000,489.000000,489.000000
mean,1826.096115,1826.449898,0.353783
std,146.414528,156.524032,57.217478
min,1258.000000,1244.000000,-159.000000
25%,1727.000000,1719.000000,-38.000000
50%,1836.000000,1833.000000,-7.000000
75%,1935.000000,1942.000000,34.000000
max,2169.000000,2234.000000,234.000000


'\nGoals stats:'

,gs,ga,matches_played
count,489.000000,489.000000,489.00000
mean,5.562372,5.562372,3.94274
std,4.530034,2.746215,1.46852
min,0.000000,0.000000,1.00000
25%,2.000000,4.000000,3.00000
50%,4.000000,5.000000,3.00000
75%,8.000000,7.000000,5.00000
max,27.000000,16.000000,7.00000


'\nNulls in next_edition   : 0'

'Nulls in next_placement : 0'

## Team Field Check

Review the team dataset's column types before building participation and confederation visualizations.

In [125]:
#display(teams.describe)
#print(teams.isna().sum())
print(teams.dtypes)

#Assign 

tournament_id        object
edition               int64
team_id              object
country              object
team_code            object
confederation        object
tournament_name      object
placement            object
position            float64
matches_played      float64
era                category
dtype: object


## Distribution of 2026 FIFA Men's World Cup Teams by Confederation

Ahead of the 2026 FIFA Men's World Cup, the tournament expanded from 32 to 48 nations. The expansion increased access for several confederations and created more opportunities for countries that had historically been under-represented.

- CAF: 5 to 10 nations
- AFC: 5 to 9 nations
- CONCACAF: 4 to 6 nations

The 2026 edition includes four debutants:

- Curacao
- Uzbekistan
- Jordan
- Cape Verde

That is the highest debutant count since 2006, when eight teams made their first World Cup appearance.

In [126]:
#tree map of all the represented nations by confederation
CONFEDERATION_COLORS = {
    "UEFA":     "#1A56DB",
    "CAF":      "#F5A623",
    "AFC":      "#E02020",
    "CONMEBOL": "#27AE60",
    "CONCACAF": "#8E44AD",
    "OFC":      "#17A8CD",
}

teams_df = teams.copy()
teams_df["edition"] = pd.to_numeric(teams_df["edition"], errors="coerce")
teams_df = teams_df.dropna(subset=["edition", "country", "confederation"]).copy()
teams_df["edition"] = teams_df["edition"].astype(int)

# Track each country's cumulative tournament appearances at each edition.
teams_df = teams_df.sort_values(["country", "edition"]).copy()
teams_df["prior_participations"] = teams_df.groupby("country").cumcount()
teams_df["participation_count"] = teams_df["prior_participations"] + 1
teams_df["is_first_timer"] = teams_df["prior_participations"].eq(0)
teams_df["participation_label"] = np.where(
    teams_df["is_first_timer"],
    "★",
    "",
)
teams_df["country_label"] = np.where(
    teams_df["is_first_timer"],
    teams_df["country"] + " ★",
    teams_df["country"],
)

editions = sorted(teams_df["edition"].unique())

# 1. Build one trace per edition.
fig = go.Figure()

for edition in editions:
    df_ed = teams_df[teams_df["edition"] == edition].copy()
    temp = px.treemap(
        df_ed,
        path=[px.Constant(f"{edition} FIFA Men's World Cup"), "confederation", "country_label"],
        color="confederation",
        color_discrete_map=CONFEDERATION_COLORS,
        custom_data=["country", "participation_count", "prior_participations", "is_first_timer"],
    )

    trace = temp.data[0]
    trace.visible = edition == editions[-1]
    trace.name = str(edition)
    trace.hovertemplate = (
        "<b>%{customdata[0]}</b><br>"
        "Participation count: %{customdata[1]}<br>"
        "Previous participations: %{customdata[2]}<br>"
        "First timer: %{customdata[3]}"
        "<extra></extra>"
    )
    fig.add_trace(trace)

# 2. Build dropdown buttons.
buttons = []
for edition in sorted(editions, reverse=True):
    i = editions.index(edition)
    df_ed = teams_df[teams_df["edition"] == edition]
    first_timers = df_ed.loc[df_ed["is_first_timer"], "country"].sort_values().tolist()
    first_timer_summary = ", ".join(first_timers) if first_timers else "None"

    buttons.append(dict(
        label=str(edition),
        method="update",
        args=[
            {"visible": [j == i for j in range(len(editions))]},
            {
                "title": {
                    "text": f"Distribution of {edition} FIFA Men's World Cup Teams by Confederation",
                    "x": 0.5,
                    "font": dict(size=20, color="#3A2A1A"),
                },
                "annotations": [
                    dict(
                        text=f"First timers: {first_timer_summary}",
                        x=0, y=1.04,
                        xref="paper", yref="paper",
                        showarrow=False,
                        align="left",
                        font=dict(size=11, color="#5A4632"),
                    ),
                    dict(
                        text="Data Source: Kaggle | @cartierkut1",
                        x=0, y=-0.03,
                        xref="paper", yref="paper",
                        showarrow=False,
                        font=dict(size=10, color="#5A4632"),
                    ),
                ],
            },
        ],
    ))

latest_edition = editions[-1]
latest_first_timers = teams_df.loc[
    teams_df["edition"].eq(latest_edition) & teams_df["is_first_timer"],
    "country",
].sort_values().tolist()
latest_first_timer_summary = ", ".join(latest_first_timers) if latest_first_timers else "None"

# 3. Layout.
fig.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True,
        x=1.0,
        xanchor="right",
        y=1.055,
        yanchor="top",
        bgcolor="#EFE3CF",
        bordercolor="#8B6914",
        font=dict(family="Inter, sans-serif", color="#3A2A1A", size=12),
    )],
    annotations=[
        dict(
            text=f"First timers: {latest_first_timer_summary}. Marked with '★' ",
            x=0, y=1.04,
            xref="paper", yref="paper",
            showarrow=False,
            align="left",
            font=dict(size=11, color="#5A4632"),
        ),
        dict(
            text="Data Source: Kaggle | @cartierkut1",
            x=0, y=-0.03,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10, color="#5A4632"),
        ),
    ],
    margin=dict(t=100, l=25, r=25, b=25),
    height=800,
    width=800,
    title=dict(
        text=f"Distribution of {latest_edition} FIFA Men's World Cup Teams by Confederation",
        x=0.5,
        font=dict(size=20, color="#3A2A1A"),
    ),
    font=dict(family="Inter, sans-serif", color="#3A2A1A", size=11),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
)

fig.update_traces(
    marker=dict(cornerradius=5),
    textposition="top left",
    insidetextfont=dict(family="Inter, sans-serif", size=12, color="#3A2A1A"),
)

fig.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "2026_distribution",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})
fig.write_image("assets/visualizations/2026_distribution.png", scale=2)


### Debutants by Edition

Count first-time World Cup teams in each edition and color the results by historical era to show how tournament expansion and qualification changes affected new entrants.

In [127]:
#World Cup debutants by edition
debutants = teams_df.copy()
debutants = (
    debutants
    .groupby(["edition", "era"], as_index=False, observed=True)
    .agg(dc=("is_first_timer", "sum"))
    .sort_values(["edition"])
    .reset_index(drop=True)
)
debutants["dc"] = debutants["dc"].astype(int)
tick_vals = sorted(debutants["edition"].unique())

era_colors = {
    "Early Era": "#7A4E2D",
    "Golden Age": "#C99700",
    "Modern Era": "#2F6F73",
    "Contemporary": "#8A3FFC",
    "Recent": "#D1495B",
}


fig = px.bar(
    debutants,
    x="edition",
    y="dc",
    color="era",
    color_discrete_map=era_colors,
    text='dc',   
)

fig.update_layout(
    height=600,
    width=1000,
    title=dict(
        text=f"Historical FIFA Men's World Cup debutants",
        x=0.5,
        font=dict(size=20, color="#3A2A1A"),
    ),
    coloraxis_colorbar=dict(title="Goals Rank"),
    font=dict(
        family="Gill Sans, sans-serif",
        color="#3A2A1A",
        size=11,
    ),
    margin=dict(l=40, r=40, t=70, b=60),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    xaxis=dict(
        title="Tournament Edition",
        tickmode="array",
        tickvals=tick_vals,
        ticktext=[str(x) for x in tick_vals],
        showgrid=False,
        zeroline=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
        tickangle=30,
    ),
    yaxis=dict(
        title="Number of Debutants",
        showgrid=False,
        zeroline=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
    ),
    annotations=[
        dict(
            text="Data Source: Kaggle | @cartierkut1",
            x=0, y=-0.1,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10, color="#5A4632"),
        ),
    ],
)

fig.update_traces(
    textposition='inside'
)

expansion_years = {
    1982: "Expanded to 24 teams",
    1998: "Expanded to 32 teams",
    2026: "Expanded to 48 teams",
}

max_debutants = debutants["dc"].max()

for year, label in expansion_years.items():
    if year in debutants["edition"].values:
        year_debutants = debutants.loc[
            debutants["edition"].eq(year),
            "dc"
        ].iloc[0]

        fig.add_annotation(
            x=year,
            y=year_debutants,
            text=label,
            showarrow=True,
            arrowhead=1,
            ax=10,
            ay=-25,
            textangle=-60,
            font=dict(size=10, color="#5A4632"),
            xanchor="left",
            yanchor="bottom",
        )


fig.update_yaxes(range=[0, max_debutants + 4])

fig.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "historical_debutants",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})


### Cumulative Participation by Confederation

Aggregate team entries by edition and confederation to compare how regional representation has changed across tournament eras.

In [128]:
#Participations per confederation at every wc
participation = teams.copy()
print(participation.columns)

p_sum = (
    participation
    .dropna(subset=["confederation", "era"])
    .groupby(["edition", "era", "confederation"], as_index=False, observed=True)
    .agg(cs=("country", "nunique"))
    .sort_values(["edition", "confederation"])
    .reset_index(drop=True)
)


era_colors = {
    "Early Era": "#7A4E2D",
    "Golden Age": "#C99700",
    "Modern Era": "#2F6F73",
    "Contemporary": "#8A3FFC",
    "Recent": "#D1495B",
}


fig = px.bar(
    p_sum,
    x="edition",
    y="cs",
    color="confederation",
    color_discrete_map=CONFEDERATION_COLORS,
    text='cs',   
)

fig.update_layout(
    height=600,
    width=1000,
    title=dict(
        text=f"Historical FIFA Men's World Cup participation by Confederation",
        x=0.5,
        font=dict(size=20, color="#3A2A1A"),
    ),
    coloraxis_colorbar=dict(title="Goals Rank"),
    font=dict(
        family="Gill Sans, sans-serif",
        color="#3A2A1A",
        size=11,
    ),
    margin=dict(l=40, r=40, t=70, b=60),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    xaxis=dict(
        title="Tournament Edition",
        tickmode="array",
        tickvals=tick_vals,
        ticktext=[str(x) for x in tick_vals],
        showgrid=False,
        zeroline=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
        tickangle=30,
    ),
    yaxis=dict(
        title="Confederation Distribution",
        showgrid=False,
        zeroline=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
    ),
    annotations=[
        dict(
            text="Data Source: Kaggle | @cartierkut1",
            x=0, y=-0.1,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10, color="#5A4632"),
        ),
    ],
)

fig.update_traces(
    textposition='outside'
)
fig.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "all_time_distribution",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})
fig.write_image("assets/visualizations/all_time_distribution.png", scale=2)


Index(['tournament_id', 'edition', 'team_id', 'country', 'team_code',
       'confederation', 'tournament_name', 'placement', 'position',
       'matches_played', 'era'],
      dtype='object')


### Confederation Participation Explorer

Build an interactive confederation view so each region's tournament participation can be inspected independently across editions.

In [129]:
#Participations per confederation at every wc
participation = teams.copy()

p_sum = (
    participation
    .dropna(subset=["confederation", "country", "era"])
    .groupby(["edition", "era", "confederation"], as_index=False, observed=True)
    .agg(cs=("country", "nunique"))
    .sort_values(["edition", "confederation"])
    .reset_index(drop=True)
)

confederations = sorted(p_sum["confederation"].unique())
tick_vals = sorted(p_sum["edition"].unique())
edition_grid = pd.DataFrame({"edition": tick_vals})
edition_era = p_sum.drop_duplicates("edition").set_index("edition")["era"]
y_axis_max = max(1, p_sum["cs"].max()) * 1.15
initial_confederation = confederations[-1]

era_colors = {
    "Early Era": "#7A4E2D",
    "Golden Age": "#C99700",
    "Modern Era": "#2F6F73",
    "Contemporary": "#8A3FFC",
    "Recent": "#D1495B",
}


def confed_annotations(confederation):
    annotations = [
        dict(
            text="Data Source: Kaggle | @cartierkut1",
            x=0, y=-0.1,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10, color="#5A4632"),
        )
    ]

    if confederation == "CAF":
        caf_1934 = p_sum[
            p_sum["confederation"].eq("CAF") &
            p_sum["edition"].eq(1934)
        ]

        if not caf_1934.empty:
            first_african_team_count = caf_1934["cs"].iloc[0]
            first_african_team_rows = participation.loc[
                participation["confederation"].eq("CAF") &
                participation["edition"].eq(1934),
                "country",
            ]
            first_african_team = (
                first_african_team_rows.iloc[0]
                if not first_african_team_rows.empty
                else "Egypt"
            )

            annotations.append(dict(
                x=1934,
                y=first_african_team_count,
                text=f"{first_african_team}: first African team",
                showarrow=True,
                arrowhead=1,
                ax=45,
                ay=-30,
                font=dict(size=12, color="#3A2A1A"),
                arrowcolor="#5A4632",
            ))

    return annotations


# 1. Build one trace per confederation.
fig = go.Figure()

for confederation in confederations:
    df_confed = edition_grid.merge(
        p_sum[p_sum["confederation"].eq(confederation)],
        on="edition",
        how="left",
    )
    df_confed["confederation"] = confederation
    df_confed["era"] = df_confed["era"].fillna(df_confed["edition"].map(edition_era))
    df_confed["cs"] = df_confed["cs"].fillna(0).astype(int)
    df_confed["bar_text"] = df_confed["cs"].where(df_confed["cs"].gt(0), "")

    fig.add_trace(go.Bar(
        x=df_confed["edition"],
        y=df_confed["cs"],
        text=df_confed["bar_text"],
        name=str(confederation),
        visible=confederation == initial_confederation,
        width=2.4,
        marker_color=df_confed["era"].map(era_colors),
        customdata=df_confed[["confederation", "era"]],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Era: %{customdata[1]}<br>"
            "Edition: %{x}<br>"
            "Participation count: %{y}"
            "<extra></extra>"
        ),
    ))

# 2. Build dropdown buttons.
buttons = []
for confederation in sorted(confederations, reverse=True):
    i = confederations.index(confederation)

    buttons.append(dict(
        label=str(confederation),
        method="update",
        args=[
            {"visible": [j == i for j in range(len(confederations))]},
            {
                "title": {
                    "text": f"Historical participation count for {confederation} nations at FIFA Men's World Cup",
                    "x": 0.5,
                    "font": dict(size=20, color="#3A2A1A"),
                },
                "annotations": confed_annotations(confederation),
            },
        ],
    ))

# 3. Layout.
fig.update_layout(
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        showactive=True,
        x=1.0,
        xanchor="right",
        y=1.085,
        yanchor="top",
        bgcolor="#EFE3CF",
        bordercolor="#8B6914",
        font=dict(family="Inter, sans-serif", color="#3A2A1A", size=12),
    )],
    xaxis=dict(
        title="Tournament Edition",
        tickmode="array",
        tickvals=tick_vals,
        ticktext=[str(x) for x in tick_vals],
        showgrid=False,
        showline=True,
        zeroline=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
        tickangle=30,
        range=[min(tick_vals) - 2, max(tick_vals) + 2],
    ),
    yaxis=dict(
        title="Participant count",
        showgrid=False,
        zeroline=False,
        showline=False,
        showticklabels=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
        range=[0, y_axis_max],
    ),
    annotations=confed_annotations(initial_confederation),
    margin=dict(t=100, l=25, r=25, b=25),
    height=600,
    width=900,
    title=dict(
        text=f"Historical participation count for {initial_confederation} nations at FIFA Men's World Cup",
        x=0.5,
        font=dict(size=20, color="#3A2A1A"),
    ),
    font=dict(family="Inter, sans-serif", color="#3A2A1A", size=11),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
)

fig.update_traces(
    textposition="outside",
    insidetextfont=dict(family="Inter, sans-serif", size=12, color="#3A2A1A"),
)

fig.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "participation_by_confederation",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})
fig.write_image(f"assets/visualizations/{initial_confederation}_participation.png", height= 800, width= 1200, scale=3)


## Goal Production Analysis

Create a goals table from match results, rank teams by goals scored and conceded within each tournament, and attach placement outcomes for later comparisons.

In [130]:
#Add goal tally across World Cups
results_df = results.copy()

placement_score_map = {
    "Winner": 7,
    "Runner-up": 6,
    "Third Place": 5,
    "Fourth Place": 4,
    "Semi-final": 4,
    "Quarter-final": 3,
    "Round of 16": 2,
    "Group Stage": 1,
}

placement["placement_score"] = placement["placement"].map(placement_score_map)

display(placement[["edition", "country", "placement", "position", "placement_score"]])

matches = (
    results_df.groupby(["edition", "era", "tournament_id"], dropna=False, as_index=False, observed=True)
    .agg(
        total_matches=("match_id", "nunique"),
        max_match_number=("match_number", "max"),
    )
)

gls = (
    results_df.groupby(["edition", "era", "tournament_id", "country"], dropna=False, as_index=False, observed=True)
    .agg(
        gf=("team_score", "sum"),
        ga=("opponent_score", "sum"),
        team_matches=("match_id", "nunique"),
    )
)

gls["gls_rank"] = (
    gls.groupby(["edition", "tournament_id"])["gf"]
    .rank(method="min", ascending=False)
    .astype(int)
)

gls["ga_rank"] = (
    gls.groupby(["edition", "tournament_id"])["ga"]
    .rank(method="min", ascending=True)
    .astype(int)
)

gls = gls.sort_values(["edition", "country"]).reset_index(drop=True)

#Add placements
gls = gls.merge(
    placement[["country", "placement", "edition", "position"]],
                on=["country", "edition"],
                how="left")
gls = gls.merge(
    matches[["edition", "total_matches"]],
                on=["edition"],
                how="left")

display(matches)
display(gls)



,edition,country,placement,position,placement_score
0,1930,Uruguay,Winner,1,7
1,1930,Argentina,Runner-up,2,6
2,1930,United States,Third Place,3,5
3,1930,Yugoslavia,Fourth Place,4,4
4,1930,Belgium,Group Stage,13,1
...,...,...,...,...,...
484,2022,Saudi Arabia,Group Stage,32,1
485,2022,Serbia,Group Stage,32,1
486,2022,Tunisia,Group Stage,32,1
487,2022,Uruguay,Group Stage,32,1


,edition,era,tournament_id,total_matches,max_match_number
0,1930,Early Era,WC-1930,18,18
1,1934,Early Era,WC-1934,17,17
2,1938,Early Era,WC-1938,18,18
3,1950,Early Era,WC-1950,22,22
4,1954,Golden Age,WC-1954,26,26
5,1958,Golden Age,WC-1958,35,35
6,1962,Golden Age,WC-1962,32,32
7,1966,Golden Age,WC-1966,32,32
8,1970,Golden Age,WC-1970,32,32
9,1974,Modern Era,WC-1974,38,38


,edition,era,tournament_id,country,gf,ga,team_matches,gls_rank,ga_rank,placement,position,total_matches
0,1930,Early Era,WC-1930,Argentina,18,9,5,1,12,Runner-up,2,18
1,1930,Early Era,WC-1930,Belgium,0,4,2,12,6,Group Stage,13,18
2,1930,Early Era,WC-1930,Bolivia,0,8,2,12,11,Group Stage,13,18
3,1930,Early Era,WC-1930,Brazil,5,2,2,5,1,Group Stage,13,18
4,1930,Early Era,WC-1930,Chile,5,3,3,5,2,Group Stage,13,18
...,...,...,...,...,...,...,...,...,...,...,...,...
484,2022,Recent,WC-2022,Switzerland,5,9,4,11,31,Round of 16,16,64
485,2022,Recent,WC-2022,Tunisia,1,1,3,28,1,Group Stage,32,64
486,2022,Recent,WC-2022,United States,3,4,4,21,9,Round of 16,16,64
487,2022,Recent,WC-2022,Uruguay,2,2,3,25,2,Group Stage,32,64


### Goal Table Columns

Confirm the derived goals dataset includes the expected fields before using it for country and tournament-level charts.

In [131]:
print(gls.columns)

Index(['edition', 'era', 'tournament_id', 'country', 'gf', 'ga',
       'team_matches', 'gls_rank', 'ga_rank', 'placement', 'position',
       'total_matches'],
      dtype='object')


### Goals by Country

Use an interactive country selector to view how many goals each nation scored across World Cup editions.

In [132]:
#Goals Per Country at every WC


countries = sorted(gls["country"].dropna().unique())
tick_vals = sorted(gls["edition"].unique())
edition_grid = pd.DataFrame({"edition": tick_vals})
edition_era = gls.drop_duplicates("edition").set_index("edition")["era"]
initial_country = countries[-1]

era_colors = {
    "Early Era": "#7A4E2D",
    "Golden Age": "#C99700",
    "Modern Era": "#2F6F73",
    "Contemporary": "#8A3FFC",
    "Recent": "#D1495B",
}


def country_goal_df(country):
    df_country = edition_grid.merge(
        gls[gls["country"].eq(country)],
        on="edition",
        how="left",
    )
    df_country["country"] = country
    df_country["era"] = df_country["era"].fillna(df_country["edition"].map(edition_era))
    df_country["gf"] = df_country["gf"].fillna(0).astype(int)
    df_country["team_matches"] = df_country["team_matches"].fillna(0).astype(int)
    df_country["bar_text"] = df_country["gf"].where(df_country["gf"].gt(0), "")
    return df_country


def build_country_goal_fig(country):
    df_country = country_goal_df(country)

    country_fig = go.Figure()

    for era, color in era_colors.items():
        df_era = df_country[df_country["era"].eq(era)]
        country_fig.add_bar(
            x=df_era["edition"],
            y=df_era["gf"],
            text=df_era["bar_text"],
            name=era,
            width=2.4,
            marker_color=color,
            customdata=df_era[["country", "era", "team_matches"]],
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Era: %{customdata[1]}<br>"
                "Edition: %{x}<br>"
                "Goals tally: %{y}<br>"
                "Team matches: %{customdata[2]}"
                "<extra></extra>"
            ),
        )

    country_fig.update_layout(
        xaxis=dict(
            title="Tournament Edition",
            tickmode="array",
            tickvals=tick_vals,
            ticktext=[str(x) for x in tick_vals],
            showgrid=False,
            showline=True,
            zeroline=False,
            linecolor="#5A4632",
            tickfont=dict(size=13, color="#5A4632"),
            tickangle=30,
            range=[min(tick_vals) - 2, max(tick_vals) + 2],
        ),
        yaxis=dict(
            title="Goals Tally",
            showgrid=False,
            zeroline=False,
            showline=False,
            showticklabels=False,
            linecolor="#5A4632",
            tickfont=dict(size=13, color="#5A4632"),
        ),
        annotations=[
            dict(
                text="Data Source: Kaggle | @cartierkut1",
                x=0, y=-0.1,
                xref="paper", yref="paper",
                showarrow=False,
                font=dict(size=10, color="#5A4632"),
            )
        ],
        margin=dict(t=100, l=25, r=25, b=25),
        showlegend=True,
        height=600,
        width=900,
        title=dict(
            text=f"Historical goals tally for {country} at FIFA Men's World Cup",
            x=0.5,
            font=dict(size=20, color="#3A2A1A"),
        ),
        font=dict(family="Inter, sans-serif", color="#3A2A1A", size=11),
        paper_bgcolor="#EFE3CF",
        plot_bgcolor="#EFE3CF",
    )

    country_fig.update_traces(
        textposition="outside",
        insidetextfont=dict(family="Inter, sans-serif", size=12, color="#3A2A1A"),
    )

    return country_fig


def show_country_goals(country):
    if country not in countries:
        return

    selected_fig = build_country_goal_fig(country)
    selected_fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": f"{country}_goal_tally",
            "height": 800,
            "width": 1200,
            "scale": 3,
        }
    })


country_picker = widgets.Combobox(
    value=initial_country,
    placeholder="Search country",
    options=countries,
    description="Country:",
    ensure_option=True,
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)

chart_output = widgets.Output()


def update_country_chart(change=None):
    with chart_output:
        clear_output(wait=True)
        show_country_goals(country_picker.value)


country_picker.observe(update_country_chart, names="value")
display(widgets.VBox([country_picker, chart_output]))
update_country_chart()

build_country_goal_fig(initial_country).write_image(
    f"assets/visualizations/{initial_country}_goal_tally.png",
    height=800,
    width=1200,
    scale=3,
)


### Total Goals by Tournament

Aggregate team goals and total matches into tournament totals to compare scoring volume across editions and eras.

In [144]:
#View all the goals scored at every world cup
goals_by_tournament = (
    gls.groupby(["edition", "era"], as_index=False, observed=True)
    .agg(
        total_goals=("gf", "sum"),
        total_matches=("total_matches", "first"),
    )
)
tick_vals = sorted(goals_by_tournament["edition"].unique())

goals_by_tournament['gpg'] = (goals_by_tournament['total_goals'] / goals_by_tournament['total_matches']).round(2)


fig = px.bar(
    goals_by_tournament,
    x="edition",
    y="total_goals",
    color="era",
    color_discrete_map=era_colors,
    text='total_goals',
    hover_data={"total_matches": True}
)


fig.update_layout(
    height=600,
    width=1000,
     title=dict(
            font=dict(size=20, color="#3A2A1A"),
            text= f"Total goals scored at every World Cup",
            x=0.5
        ),
    legend = dict(
        title="Team"
        ),
    font=dict(
        family="Gill Sans, Arial, sans-serif",
        color="#3A2A1A",
        size=10,
    ),
    margin=dict(l=40, r=40, t=60, b=40),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    xaxis=dict(
        title="Tournament Edition",
        showgrid=False,
        gridcolor='rgba(0,255,65,0.15)',
        zeroline=False,
        linecolor='#5A4632',
        tickfont=dict(size=13, color="#5A4632"),
        tickmode="array",
        tickvals=tick_vals,
        ticktext=[str(x) for x in tick_vals],
    ),
    yaxis=dict(
        title="Goals Scored",
        showgrid=False,
        gridcolor='rgba(0,255,65,0.15)',        
        showline=False,
        showticklabels=False,
        zeroline=False,
        linecolor='#5A4632',
        tickfont=dict(size=13, color="#5A4632"),
    ),
    annotations=[dict(
        text="Data Source: Kaggle | @cartierkut1",
        xref="paper", yref="paper",
        x=0, y=-0.1,
        showarrow=False,
        font=dict(size=10, color="#5A4632"),
        )]
        
)

fig.update_traces(
    textposition='outside'
)



fig.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "total_goals_by_world_cup",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})

fig2 = px.bar(
    goals_by_tournament,
    x="edition",
    y="gpg",
    color="era",
    color_discrete_map=era_colors,
    text='gpg',
    hover_data={"total_matches": True}
)


fig2.update_layout(
    height=600,
    width=1000,
     title=dict(
            font=dict(size=20, color="#3A2A1A"),
            text= f"Total goals scored at every World Cup",
            x=0.5
        ),
    legend = dict(
        title="Team"
        ),
    font=dict(
        family="Gill Sans, Arial, sans-serif",
        color="#3A2A1A",
        size=10,
    ),
    margin=dict(l=40, r=40, t=60, b=40),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    xaxis=dict(
        title="Tournament Edition",
        showgrid=False,
        gridcolor='rgba(0,255,65,0.15)',
        zeroline=False,
        linecolor='#5A4632',
        tickfont=dict(size=13, color="#5A4632"),
        tickmode="array",
        tickvals=tick_vals,
        ticktext=[str(x) for x in tick_vals],
    ),
    yaxis=dict(
        title="Goals Scored",
        showgrid=False,
        gridcolor='rgba(0,255,65,0.15)',        
        showline=False,
        showticklabels=False,
        zeroline=False,
        linecolor='#5A4632',
        tickfont=dict(size=13, color="#5A4632"),
    ),
    annotations=[dict(
        text="Data Source: Kaggle | @cartierkut1",
        xref="paper", yref="paper",
        x=0, y=-0.1,
        showarrow=False,
        font=dict(size=10, color="#5A4632"),
        )]
        
)

fig2.update_traces(
    textposition='outside'
)



fig2.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "total_goals_by_world_cup",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})


## Winner Follow-Up Performance

Track how World Cup winners performed in the immediate next edition, treating missing next-edition appearances as did-not-qualify cases and excluding 1930 for a fairer tournament-format comparison.

### Winner Follow-Up View

Rebuild the winner follow-up dataset for a second visualization focused on next-edition placement and qualification status.

In [134]:
base = placement.copy()
base["edition"] = pd.to_numeric(base["edition"])
base["position"] = pd.to_numeric(base["position"], errors="coerce")

editions = sorted(base["edition"].unique())
next_edition_map = dict(zip(editions[:-1], editions[1:]))

# winners only
df = base.loc[base["placement"].eq("Winner") & (base["teams"] > 15), ["edition", "country"]].copy()

#min year
min_ed = df["edition"].min()

df["next_edition"] = df["edition"].map(next_edition_map)

# lookup that checks the same team in the immediate next World Cup
lookup = (
    base[["country", "edition", "placement", "position"]]
    .rename(
        columns={
            "edition": "next_edition",
            "placement": "next_placement",
            "position": "next_position",
        }
    )
)

df = df.merge(lookup, on=["country", "next_edition"], how="left")

# missing in the immediate next edition = did not qualify
df.loc[df["next_edition"].notna() & df["next_position"].isna(), "next_placement"] = "DNQ"

# no next edition available in the dataset yet
df.loc[df["next_edition"].isna(), "next_placement"] = "TBD"

dnq_rank = base["position"].max() + 1
df["plot_position"] = df["next_position"]
df.loc[df["next_placement"].eq("DNQ"), "plot_position"] = dnq_rank

placement_map = {
    "Winner": "W",
    "Runner-up": "F",
    "Third Place": "3P",
    "Fourth Place": "4P",
    "Semi-final": "SF",
    "Quarter-final": "QF",
    "Round of 16": "R16",
    "Group Stage": "GS",
    "DNQ": "DNQ",
    "TBD": "TBD",
}

value_map = {
    "W": 1,
    "F": 2,
    "3P": 3,
    "4P": 4,
    "QF": 8,
    "R16": 16,
    "GS": 32,
    }

df["next_placement_short"] = df["next_placement"].map(placement_map)
df["next_placement_value"] = df["next_placement_short"].map(value_map)
fig = px.bar(
    df,
    x="edition",
    y="plot_position",
    hover_data=["country", "next_edition", "next_placement"],
    text="next_placement_short",
    title="Every World Cup Winners Placement the following Edition",
    color="country",
)
fig.update_yaxes(
    tickvals=[1, 2, 3, 4, 8, 16, 32, dnq_rank],
    ticktext=["W", "F", "3P", "4P", "QF", "R16", "GS", "DNQ"],
    )

fig.update_layout(
    height=600,
    width=1000,
    title=dict(
        text=f"World Cup Winners Placement the following Edition since {min_ed}",
        x=0.5,
        font=dict(size=20, color="#3A2A1A"),
    ),
    coloraxis_colorbar=dict(title="Goals Rank"),
    font=dict(
        family="Gill Sans, sans-serif",
        color="#3A2A1A",
        size=11,
    ),
    margin=dict(l=40, r=40, t=70, b=60),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    xaxis=dict(
        title="Tournament Edition",
        tickmode="array",
        tickvals=tick_vals,
        ticktext=[str(x) for x in tick_vals],
        showgrid=False,
        zeroline=False,
        showline=False,
        side="bottom",
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
    ),
    yaxis=dict(
        title="Next Placement",
        autorange="reversed",
        showgrid=False,
        zeroline=False,        
        showline=False,
        showticklabels=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
    ),
    annotations=[
        dict(
            text="Data Source: Kaggle | @cartierkut1",
            x=0, y=-0.1,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10, color="#5A4632"),
        ),
    ],
)

fig.update_traces(
    textposition='outside'
)


fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "winners_next_placements",
            "height": 800,
            "width": 1200,
            "scale": 3,
        }
    })
fig.write_image("assets/visualizations/winners_next_placements.png", scale=3)

## Host Nation Effect

looking at the impact of being a host nation and how it affects tournament performance

In [135]:
#looking at the placement of every host nation
HOST_NAME_ALIASES = {
    "usa": "united states",
    "u.s.a.": "united states",
    "united states of america": "united states",
    "korea republic": "south korea",
    "republic of korea": "south korea",
    "west germany": "germany",
}


def normalize_country_name(value):
    normalized = str(value).strip().lower()
    return HOST_NAME_ALIASES.get(normalized, normalized)


def parse_host_countries(value):
    host_text = str(value)
    for separator in [" / ", "/", " & ", " and ", ","]:
        host_text = host_text.replace(separator, ",")
    return {
        normalize_country_name(host_country)
        for host_country in host_text.split(",")
        if host_country.strip()
    }


placement["is_host"] = placement.apply(
    lambda row: normalize_country_name(row["country"]) in parse_host_countries(row["host"]),
    axis=1,
)

hosts = placement.loc[placement["is_host"]].copy()

#replace null next placements with DNQ
print(hosts["next_placement"].isnull().sum())
hosts[["next_placement", "next_edition" ]] = hosts[["next_placement", "next_edition" ]].fillna("DNQ")
hosts["next_position"] = hosts["next_position"].fillna(0)

#print the average placement position
host_finish = hosts["position"].mean()
host_goals = hosts["gs"].mean()
host_concede = hosts["ga"].mean()



placement_bins = [0, 1, 2, 4, 8, 16, 32, 48]
placement_labels = [
    "Winner",
    "Runner-up",
    "Semi-final",
    "Quarter-final",
    "Round of 16",
    "Group Stage",
    "Expanded Field",
]

finish_place = pd.cut(
    [host_finish],
    bins=placement_bins,
    labels=placement_labels,
    include_lowest=True,
)[0]

print(f"Average host finish: {host_finish:.2f}")
print(f"Average host placement: {finish_place}")
print(f"Average host goals scored: {host_goals:.2f}")
print(f"Average host goals conceded: {host_concede:.2f}")
display(hosts)


0
Average host finish: 7.74
Average host placement: Quarter-final
Average host goals scored: 9.65
Average host goals conceded: 5.22


C:\Users\oukan\AppData\Local\Temp\ipykernel_9736\3751282049.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  hosts[["next_placement", "next_edition" ]] = hosts[["next_placement", "next_edition" ]].fillna("DNQ")


,edition,tournament_id,year,country,team_id,team_code,confederation,placement,position,matches_played,gs,ga,start_elo,finish_elo,elo_change,next_edition,next_placement,next_position,teams,host,elo_rank,placement_score,is_host
0,1930,WC-1930,1930,Uruguay,URY,URY,CONMEBOL,Winner,1,4,15,3,1968,2034,66,1950,Winner,1.0,13,Uruguay,2,7,True
13,1934,WC-1934,1934,Italy,ITA,ITA,UEFA,Winner,1,5,12,3,1975,2047,72,1938,Winner,1.0,16,Italy,3,7,True
35,1938,WC-1938,1938,France,FRA,FRA,UEFA,Quarter-final,8,2,4,4,1630,1642,12,1954,Group Stage,16.0,15,France,10,3,True
45,1950,WC-1950,1950,Brazil,BRA,BRA,CONMEBOL,Runner-up,2,6,22,6,1967,2013,46,1954,Quarter-final,8.0,13,Brazil,4,6,True
63,1954,WC-1954,1954,Switzerland,CHE,CHE,UEFA,Quarter-final,8,4,11,11,1673,1722,49,1962,Group Stage,16.0,16,Switzerland,15,3,True
74,1958,WC-1958,1958,Sweden,SWE,SWE,UEFA,Runner-up,2,6,12,7,1803,1900,97,1970,Group Stage,16.0,16,Sweden,12,6,True
91,1962,WC-1962,1962,Chile,CHL,CHL,CONMEBOL,Third Place,3,6,10,8,1740,1833,93,1966,Group Stage,16.0,16,Chile,14,5,True
105,1966,WC-1966,1966,England,ENG,ENG,UEFA,Winner,1,6,11,3,1991,2046,55,1970,Quarter-final,8.0,16,England,3,7,True
126,1970,WC-1970,1970,Mexico,MEX,MEX,CONCACAF,Quarter-final,8,4,6,4,1703,1722,19,1978,Group Stage,16.0,16,Mexico,12,3,True
142,1974,WC-1974,1974,Germany,GER,DEU,UEFA,Quarter-final,8,7,13,4,2112,2144,32,1978,Quarter-final,8.0,16,West Germany,2,3,True


In [136]:
hosts.loc[hosts["next_placement"].eq("DNQ"), "plot_position"] = dnq_rank

placement_map = {
    "Winner": "W",
    "Runner-up": "F",
    "Third Place": "3P",
    "Fourth Place": "4P",
    "Semi-final": "SF",
    "Quarter-final": "QF",
    "Round of 16": "R16",
    "Group Stage": "GS",
    "DNQ": "DNQ",
    "TBD": "TBD",
}

value_map = {
    "W": 1,
    "F": 2,
    "3P": 3,
    "4P": 4,
    "QF": 8,
    "R16": 16,
    "GS": 32,
    }

hosts["placement_short"] = hosts["placement"].map(placement_map)
hosts["placement_value"] = hosts["placement_short"].map(value_map)
fig = px.bar(
    hosts,
    x="edition",
    y="position",
    hover_data=["edition", "country", "confederation", "placement", "gs", "ga"],
    text="placement_short",
    color="country",
)
fig.update_yaxes(
    tickvals=[1, 2, 3, 4, 8, 16, 32, dnq_rank],
    ticktext=["W", "F", "3P", "4P", "QF", "R16", "GS", "DNQ"],
    )

fig.update_layout(
    height=600,
    width=1000,
    title=dict(
        text=f"Historical FIFA Men's World Cup host placement",
        x=0.5,
        font=dict(size=20, color="#3A2A1A"),
    ),
    coloraxis_colorbar=dict(title="placement Rank"),
    font=dict(
        family="Gill Sans, sans-serif",
        color="#3A2A1A",
        size=11,
    ),
    margin=dict(l=40, r=40, t=70, b=60),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    xaxis=dict(
        title="Tournament Edition",
        tickmode="array",
        tickvals=tick_vals,
        ticktext=[str(x) for x in tick_vals],
        showgrid=False,
        zeroline=False,
        showline=False,
        side="bottom",
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
    ),
    yaxis=dict(
        title="Placement",
        autorange="reversed",
        showgrid=False,
        zeroline=False,        
        showline=False,
        showticklabels=False,
        linecolor="#5A4632",
        tickfont=dict(size=13, color="#5A4632"),
    ),
    annotations=[
        dict(
            text="Data Source: Kaggle | @cartierkut1",
            x=0, y=-0.1,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10, color="#5A4632"),
        ),
    ],
)

fig.update_traces(
    textposition='outside'
)


fig.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "winners_next_placements",
            "height": 800,
            "width": 1200,
            "scale": 3,
        }
    })

In [137]:
#scatter showing the correlation between elo and placement for hosts
# Build a plotting copy so chart-only columns do not alter the main hosts dataframe.
hosts_plot = hosts.sort_values(["position", "start_elo", "edition"]).copy()
hosts_plot["gd"] = hosts_plot["gs"] - hosts_plot["ga"]
hosts_plot["label"] = hosts_plot["team_id"] + "<br>" + hosts_plot["edition"].astype(str)

# Set the average Elo and average finishing position used to split the quadrants.
x_mean = hosts_plot["start_elo"].mean()
y_mean = hosts_plot["position"].mean()

# Pad the axis limits so labels and quadrant annotations have room to breathe.
x_min, x_max = hosts_plot["start_elo"].min(), hosts_plot["start_elo"].max()
y_min, y_max = hosts_plot["position"].min(), hosts_plot["position"].max()
x_pad = 50
y_pad = 1.5

x_lower, x_upper = x_min - x_pad, x_max + x_pad
y_lower, y_upper = max(0.5, y_min - y_pad), y_max + y_pad

elo_ticks = list(range(
    int(x_lower // 100 * 100),
    int(x_upper // 100 * 100 + 101),
    100,
))

# Lower position is better, so the strong-finish quadrants are above the mean line.
hosts_plot["quadrant"] = np.select(
    [
        (hosts_plot["start_elo"] >= x_mean) & (hosts_plot["position"] <= y_mean),
        (hosts_plot["start_elo"] < x_mean) & (hosts_plot["position"] <= y_mean),
        (hosts_plot["start_elo"] >= x_mean) & (hosts_plot["position"] > y_mean),
        (hosts_plot["start_elo"] < x_mean) & (hosts_plot["position"] > y_mean),
    ],
    [
        "High Elo / strong finish",
        "Low Elo / strong finish",
        "High Elo / weak finish",
        "Low Elo / weak finish",
    ],
    default="Average line",
)

# Cycle text positions to reduce label collisions around nearby points.
label_positions = [
    "top center",
    "bottom center",
    "middle right",
    "middle left",
    "top right",
    "bottom right",
    "top left",
    "bottom left",
]
hosts_plot["label_position"] = [
    label_positions[i % len(label_positions)]
    for i in range(len(hosts_plot))
]

# Keep quadrant colors consistent between markers, legend, and background shading.
quadrant_colors = {
    "High Elo / strong finish": "#2E8B57",
    "Low Elo / strong finish": "#C99700",
    "High Elo / weak finish": "#C44E52",
    "Low Elo / weak finish": "#6E6259",
}

# Draw the scatter, using quadrant as the analytical grouping and Elo rank as marker size.
fig = px.scatter(
    hosts_plot,
    x="start_elo",
    y="position",
    color="quadrant",
    size="elo_rank",
    color_discrete_map=quadrant_colors,
    hover_data={
        "country": True,
        "edition": True,
        "confederation": True,
        "placement": True,
        "start_elo": ":,.0f",
        "elo_rank": True,
        "position": False,
        "gs": True,
        "ga": True,
        "gd": True,
        "quadrant": False,
    },
    text="label",
    category_orders={"quadrant": list(quadrant_colors)},
)

# Apply per-point label placement after Plotly Express creates one trace per quadrant.
for trace in fig.data:
    for quadrant in hosts_plot["quadrant"].unique():
        text_positions = hosts_plot.loc[
            hosts_plot["quadrant"].eq(quadrant),
            "label_position",
        ].tolist()
        fig.update_traces(
            selector={"name": quadrant},
            textposition=text_positions,
            textfont=dict(size=10, color="#3A2A1A"),
            marker=dict(size=11, line=dict(width=1, color="#EFE3CF")),
            cliponaxis=False,
        )

# Quadrant background rectangles. y uses rank-like position, so lower values are stronger.
quadrant_backgrounds = [
    (x_mean, x_upper, y_lower, y_mean, "#2E8B57", 0.12),  # high Elo, strong finish
    (x_lower, x_mean, y_lower, y_mean, "#C99700", 0.12),  # low Elo, strong finish
    (x_mean, x_upper, y_mean, y_upper, "#C44E52", 0.10),  # high Elo, weak finish
    (x_lower, x_mean, y_mean, y_upper, "#6E6259", 0.08),  # low Elo, weak finish
]
for x0, x1, y0, y1, color, opacity in quadrant_backgrounds:
    fig.add_shape(
        type="rect",
        x0=x0, x1=x1, y0=y0, y1=y1,
        fillcolor=color,
        opacity=opacity,
        line_width=0,
        layer="below",
    )

# Add average lines to make the four quadrants explicit.
fig.add_vline(
    x=x_mean,
    line_dash="dash",
    line_color="#5A4632",
    line_width=1,
)
fig.add_hline(
    y=y_mean,
    line_dash="dash",
    line_color="#5A4632",
    line_width=1,
)

# Label each quadrant and the two average reference lines.
quadrant_annotations = [
    dict(
        x=(x_mean + x_upper) / 2,
        y=(y_lower + y_mean) / 2,
        text="High Elo<br>strong finish",
        showarrow=False,
        font=dict(size=12, color="#2F4A36"),
        align="center",
    ),
    dict(
        x=(x_lower + x_mean) / 2,
        y=(y_lower + y_mean) / 2,
        text="Low Elo<br>strong finish",
        showarrow=False,
        font=dict(size=12, color="#6A5600"),
        align="center",
    ),
    dict(
        x=(x_mean + x_upper) / 2,
        y=(y_mean + y_upper) / 2,
        text="High Elo<br>weak finish",
        showarrow=False,
        font=dict(size=12, color="#6A3332"),
        align="center",
    ),
    dict(
        x=(x_lower + x_mean) / 2,
        y=(y_mean + y_upper) / 2,
        text="Low Elo<br>weak finish",
        showarrow=False,
        font=dict(size=12, color="#4A403A"),
        align="center",
    ),
    dict(
        text=f"Average Elo: {x_mean:,.0f}",
        x=x_mean,
        y=y_upper,
        xanchor="left",
        yanchor="bottom",
        showarrow=False,
        font=dict(size=10, color="#5A4632"),
    ),
    dict(
        text=f"Average finish: {y_mean:.1f}",
        x=x_upper,
        y=y_mean,
        xanchor="right",
        yanchor="bottom",
        showarrow=False,
        font=dict(size=10, color="#5A4632"),
    ),
    dict(
        text="Data Source: Kaggle | @cartierkut1",
        x=0,
        y=-0.14,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(size=10, color="#5A4632"),
    ),
]

# Final visual styling and export settings.
fig.update_layout(
    height=650,
    width=1050,
    title=dict(
        text="Host Elo vs FIFA Men's World Cup Finish",
        x=0.5,
        font=dict(size=20, color="#3A2A1A"),
    ),
    font=dict(
        family="Gill Sans, sans-serif",
        color="#3A2A1A",
        size=11,
    ),
    margin=dict(l=70, r=40, t=80, b=85),
    paper_bgcolor="#EFE3CF",
    plot_bgcolor="#EFE3CF",
    xaxis=dict(
        title="Starting Elo Rating",
        range=[x_lower, x_upper],
        tickmode="array",
        tickvals=elo_ticks,
        ticktext=[str(x) for x in elo_ticks],
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="#5A4632",
        tickfont=dict(size=12, color="#5A4632"),
    ),
    yaxis=dict(
        title="Tournament Finish",
        range=[y_upper, y_lower],
        showgrid=False,
        zeroline=False,
        showline=True,
        tickmode="array",
        tickvals=sorted(hosts_plot["position"].dropna().unique()),
        ticktext=[
            hosts_plot.loc[hosts_plot["position"].eq(pos), "placement"].iloc[0]
            for pos in sorted(hosts_plot["position"].dropna().unique())
        ],
        linecolor="#5A4632",
        tickfont=dict(size=12, color="#5A4632"),
    ),
    annotations=quadrant_annotations,
)

fig.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "host_elo_finish_quadrants",
        "height": 900,
        "width": 1400,
        "scale": 3,
    }
})


## Correlation Analysis

The final sections test how strongly historical performance, Elo, goals, hosting, and recent World Cup form relate to tournament outcomes. Rolling features are shifted so a team's current tournament result is not leaked into its pre-tournament history.

In [138]:
# World Cup outcome correlation analysis

placement_score_map = {
    "Winner": 7,
    "Runner-up": 6,
    "Third Place": 5,
    "Fourth Place": 4,
    "Semi-final": 4,
    "Quarter-final": 3,
    "Round of 16": 2,
    "Group Stage": 1,
}

placement_files = sorted(Path("data/processed/world_cup").glob("[0-9][0-9][0-9][0-9]/placement.csv"))
placement_frames = []
for placement_file in placement_files:
    edition = int(placement_file.parent.name)
    edition_df = pd.read_csv(placement_file)
    edition_df["edition"] = edition
    placement_frames.append(edition_df)

outcome_df = pd.concat(placement_frames, ignore_index=True)
outcome_df = outcome_df.sort_values(["country", "edition"]).reset_index(drop=True)

# Outcome: higher is better, normalized within each tournament size.
outcome_df["edition_team_count"] = outcome_df.groupby("edition")["country"].transform("count")
outcome_df["position"] = pd.to_numeric(outcome_df["position"], errors="coerce")
outcome_df["edition_finish_scale"] = outcome_df.groupby("edition")["position"].transform("max")
outcome_df["edition_finish_scale"] = outcome_df[["edition_team_count", "edition_finish_scale"]].max(axis=1)
outcome_df["finish_score"] = 1 - (
    (outcome_df["position"] - 1) / (outcome_df["edition_finish_scale"] - 1)
)
outcome_df["placement_score"] = outcome_df["placement"].map(placement_score_map)

# In-tournament explanatory features. These describe outcomes but should not be treated as pre-tournament predictors.
for column in ["matches_played", "gs", "ga", "start_elo", "finish_elo", "elo_change"]:
    outcome_df[column] = pd.to_numeric(outcome_df[column], errors="coerce")

outcome_df["goal_difference"] = outcome_df["gs"] - outcome_df["ga"]
outcome_df["goals_per_match"] = outcome_df["gs"] / outcome_df["matches_played"]
outcome_df["goals_against_per_match"] = outcome_df["ga"] / outcome_df["matches_played"]
outcome_df["goal_difference_per_match"] = outcome_df["goal_difference"] / outcome_df["matches_played"]

# Host flag from the match results host-country column. Use pairs so co-host editions do not duplicate rows.
host_pairs = set(
    pd.read_csv("data/processed/world_cup/all_editions/results.csv")[["edition", "country"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
outcome_df["is_host"] = [
    int((int(edition), country) in host_pairs)
    for edition, country in zip(outcome_df["edition"], outcome_df["country"])
]

# Leakage-safe prior features: every rolling value is shifted so the current tournament is excluded.
team_history = outcome_df.groupby("country", group_keys=False)
outcome_df["prior_world_cup_participations"] = team_history.cumcount()
outcome_df["previous_finish_score"] = team_history["finish_score"].shift(1)
outcome_df["previous_position"] = team_history["position"].shift(1)

outcome_df["prior_avg_finish_score"] = team_history["finish_score"].transform(
    lambda series: series.shift(1).expanding().mean()
)
outcome_df["prior_best_finish_score"] = team_history["finish_score"].transform(
    lambda series: series.shift(1).expanding().max()
)
outcome_df["prior_avg_goals_per_match"] = team_history["goals_per_match"].transform(
    lambda series: series.shift(1).expanding().mean()
)
outcome_df["prior_avg_goal_diff_per_match"] = team_history["goal_difference_per_match"].transform(
    lambda series: series.shift(1).expanding().mean()
)

# Leakage-safe last-k form entering each tournament.
FORM_MATCH_WINDOW = 10
country_form_aliases = {
    "China PR": "China",
    "DR Congo": "Zaire",
}

edition_start_dates = {}
for results_file in sorted(Path("data/processed/world_cup").glob("[0-9][0-9][0-9][0-9]/results.csv")):
    edition = int(results_file.parent.name)
    edition_results = pd.read_csv(results_file, usecols=["date"])
    edition_start_dates[edition] = pd.to_datetime(edition_results["date"], errors="coerce").min()

team_result_frames = []
team_result_columns = [
    "date",
    "team",
    "team_score",
    "opponent_score",
    "result",
    "team_elo_start",
    "opponent_elo_start",
    "team_elo_delta",
]
for team_results_file in sorted(Path("data/processed/world_cup/by_confederation").glob("*/*/results.csv")):
    team_results = pd.read_csv(team_results_file, usecols=team_result_columns)
    if team_results.empty:
        continue
    team_result_frames.append(team_results)

team_results_history = pd.concat(team_result_frames, ignore_index=True)
team_results_history["date"] = pd.to_datetime(team_results_history["date"], errors="coerce")
for column in ["team_score", "opponent_score", "team_elo_start", "opponent_elo_start", "team_elo_delta"]:
    team_results_history[column] = pd.to_numeric(team_results_history[column], errors="coerce")

normalized_result = (
    team_results_history["result"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({"w": "win", "d": "draw", "l": "loss", "win": "win", "draw": "draw", "loss": "loss"})
)
score_based_result = pd.Series(
    np.select(
        [
            team_results_history["team_score"] > team_results_history["opponent_score"],
            team_results_history["team_score"] == team_results_history["opponent_score"],
            team_results_history["team_score"] < team_results_history["opponent_score"],
        ],
        ["win", "draw", "loss"],
        default=None,
    ),
    index=team_results_history.index,
    dtype="object",
)
team_results_history["normalized_result"] = normalized_result.fillna(score_based_result)
team_results_history["actual_score"] = team_results_history["normalized_result"].map({"win": 1.0, "draw": 0.5, "loss": 0.0})
team_results_history["win_indicator"] = team_results_history["normalized_result"].eq("win").astype(float)
team_results_history["goal_difference"] = team_results_history["team_score"] - team_results_history["opponent_score"]
team_results_history = team_results_history.dropna(subset=["date", "team", "team_score", "opponent_score", "actual_score"]).copy()
team_results_lookup = {
    str(team): matches.sort_values("date", kind="stable").reset_index(drop=True)
    for team, matches in team_results_history.groupby("team", sort=False)
}

def weighted_mean(values, weights):
    values = pd.to_numeric(values, errors="coerce")
    valid = values.notna()
    if not valid.any():
        return np.nan
    return float(np.average(values[valid], weights=weights[valid]))

form_rows = []
for row in outcome_df[["edition", "country"]].itertuples(index=False):
    edition_start_date = edition_start_dates.get(int(row.edition))
    lookup_country = country_form_aliases.get(str(row.country), str(row.country))
    team_matches = team_results_lookup.get(lookup_country)
    if team_matches is None or pd.isna(edition_start_date):
        recent = pd.DataFrame(columns=team_results_history.columns)
    else:
        recent = team_matches[team_matches["date"] < edition_start_date].tail(FORM_MATCH_WINDOW).copy()

    form_row = {
        "edition": int(row.edition),
        "country": str(row.country),
        "form_l10_matches": np.nan,
        "form_l10_win_pct": np.nan,
        "form_l10_goals_for": np.nan,
        "form_l10_goals_against": np.nan,
        "form_l10_goal_difference": np.nan,
        "form_l10_goals_per_match": np.nan,
        "form_l10_goals_against_per_match": np.nan,
        "form_l10_goal_difference_per_match": np.nan,
        "form_l10_elo_change": np.nan,
        "form_l10_elo_change_per_match": np.nan,
        "weighted_form_l10_result_score": np.nan,
        "weighted_form_l10_win_pct": np.nan,
        "weighted_form_l10_goals_for_per_match": np.nan,
        "weighted_form_l10_goals_against_per_match": np.nan,
        "weighted_form_l10_goal_difference_per_match": np.nan,
        "weighted_form_l10_elo_change_per_match": np.nan,
        "form_l10_first_match_date": pd.NaT,
        "form_l10_last_match_date": pd.NaT,
    }

    if not recent.empty:
        match_count = len(recent)
        weights = pd.Series(np.arange(1, match_count + 1, dtype=float), index=recent.index)
        goals_for = recent["team_score"].sum()
        goals_against = recent["opponent_score"].sum()
        goal_difference = goals_for - goals_against
        elo_change = recent["team_elo_delta"].sum(min_count=1)

        form_row.update(
            {
                "form_l10_matches": int(match_count),
                "form_l10_win_pct": float(recent["win_indicator"].mean()),
                "form_l10_goals_for": float(goals_for),
                "form_l10_goals_against": float(goals_against),
                "form_l10_goal_difference": float(goal_difference),
                "form_l10_goals_per_match": float(goals_for / match_count),
                "form_l10_goals_against_per_match": float(goals_against / match_count),
                "form_l10_goal_difference_per_match": float(goal_difference / match_count),
                "form_l10_elo_change": float(elo_change) if pd.notna(elo_change) else np.nan,
                "form_l10_elo_change_per_match": float(recent["team_elo_delta"].mean()) if recent["team_elo_delta"].notna().any() else np.nan,
                "weighted_form_l10_result_score": weighted_mean(recent["actual_score"], weights),
                "weighted_form_l10_win_pct": weighted_mean(recent["win_indicator"], weights),
                "weighted_form_l10_goals_for_per_match": weighted_mean(recent["team_score"], weights),
                "weighted_form_l10_goals_against_per_match": weighted_mean(recent["opponent_score"], weights),
                "weighted_form_l10_goal_difference_per_match": weighted_mean(recent["goal_difference"], weights),
                "weighted_form_l10_elo_change_per_match": weighted_mean(recent["team_elo_delta"], weights),
                "form_l10_first_match_date": recent["date"].min(),
                "form_l10_last_match_date": recent["date"].max(),
            }
        )
    form_rows.append(form_row)

form_df = pd.DataFrame(form_rows)
outcome_df = outcome_df.merge(form_df, on=["edition", "country"], how="left")
outcome_df["edition_start_date"] = outcome_df["edition"].map(edition_start_dates)

form_feature_columns = [
    "form_l10_matches",
    "form_l10_win_pct",
    "form_l10_goals_for",
    "form_l10_goals_against",
    "form_l10_goal_difference",
    "form_l10_goals_per_match",
    "form_l10_goals_against_per_match",
    "form_l10_goal_difference_per_match",
    "form_l10_elo_change",
    "form_l10_elo_change_per_match",
    "weighted_form_l10_result_score",
    "weighted_form_l10_win_pct",
    "weighted_form_l10_goals_for_per_match",
    "weighted_form_l10_goals_against_per_match",
    "weighted_form_l10_goal_difference_per_match",
    "weighted_form_l10_elo_change_per_match",
]

pre_tournament_features = [
    "start_elo",
    "is_host",
    "prior_world_cup_participations",
    "prior_avg_finish_score",
    "prior_best_finish_score",
    "prior_avg_goals_per_match",
    "prior_avg_goal_diff_per_match",
    "previous_finish_score",
    "previous_position",
] + form_feature_columns

in_tournament_features = [
    "matches_played",
    "gs",
    "ga",
    "goal_difference",
    "goals_per_match",
    "goals_against_per_match",
    "goal_difference_per_match",
    "finish_elo",
    "elo_change",
]

def build_correlation_table(df, feature_columns, target="finish_score"):
    rows = []
    for feature in feature_columns:
        analysis_df = df[[feature, target]].dropna()
        rows.append(
            {
                "feature": feature,
                "n": len(analysis_df),
                "pearson_corr": analysis_df[feature].corr(analysis_df[target], method="pearson"),
                "spearman_corr": analysis_df[feature].corr(analysis_df[target], method="spearman"),
            }
        )
    corr_df = pd.DataFrame(rows)
    corr_df["abs_spearman_corr"] = corr_df["spearman_corr"].abs()
    return corr_df.sort_values("abs_spearman_corr", ascending=False).reset_index(drop=True)

pre_tournament_corr = build_correlation_table(outcome_df, pre_tournament_features)
in_tournament_corr = build_correlation_table(outcome_df, in_tournament_features)

# Sanity checks requested in the plan.
assert not outcome_df.duplicated(["edition", "country"]).any(), "Expected one row per edition + country."
assert np.allclose(outcome_df.loc[outcome_df["placement"].eq("Winner"), "finish_score"], 1.0), "Winners must score 1.0."
lowest_finish_by_edition = outcome_df.loc[outcome_df.groupby("edition")["position"].idxmax()]
assert np.allclose(lowest_finish_by_edition["finish_score"], 0.0), "Lowest position in each edition must score 0.0."

known_hosts = {
    (1966, "England"),
    (1998, "France"),
    (2010, "South Africa"),
    (2022, "Qatar"),
    (2002, "South Korea"),
    (2002, "Japan"),
}
missing_hosts = [
    f"{country} {edition}"
    for edition, country in known_hosts
    if outcome_df.loc[
        outcome_df["edition"].eq(edition) & outcome_df["country"].eq(country),
        "is_host",
    ].sum() != 1
]
assert not missing_hosts, f"Host flags missing for: {missing_hosts}"
assert outcome_df["form_l10_matches"].dropna().le(FORM_MATCH_WINDOW).all(), "Form windows must not exceed last-k size."
assert (
    outcome_df.loc[outcome_df["form_l10_last_match_date"].notna(), "form_l10_last_match_date"]
    < outcome_df.loc[outcome_df["form_l10_last_match_date"].notna(), "edition_start_date"]
).all(), "Form matches must be before the tournament start date."
assert not outcome_df.duplicated(["edition", "country"]).any(), "Expected no duplicate rows after adding form features."
excluded_correlation_features = {"edition", "country", "placement"}
assert excluded_correlation_features.isdisjoint(pre_tournament_features + in_tournament_features), "Correlation features include identifiers or categorical placement."
weighted_diff_check = outcome_df[
    outcome_df["form_l10_matches"].ge(2)
    & outcome_df["form_l10_win_pct"].notna()
    & outcome_df["weighted_form_l10_win_pct"].notna()
    & ~np.isclose(outcome_df["form_l10_win_pct"], outcome_df["weighted_form_l10_win_pct"])
]
assert not weighted_diff_check.empty, "Expected at least one mixed-form team where weighted and unweighted form differ."

spot_checks = outcome_df.set_index(["edition", "country"])
for edition, country in [(1966, "England"), (2022, "Argentina")]:
    spot_row = spot_checks.loc[(edition, country)]
    assert pd.notna(spot_row["form_l10_last_match_date"]), f"Missing form rows for {country} {edition}."
    assert spot_row["form_l10_last_match_date"] < spot_row["edition_start_date"], f"Form leakage for {country} {edition}."

print(f"Analysis rows: {len(outcome_df):,}")
print(f"Editions: {outcome_df['edition'].min()}-{outcome_df['edition'].max()} ({outcome_df['edition'].nunique()} tournaments)")

print("\nPre-tournament predictors vs finish_score")
display(pre_tournament_corr)

print("\nIn-tournament explanatory stats vs finish_score")
display(in_tournament_corr)

fig_pre = px.bar(
    pre_tournament_corr,
    x="spearman_corr",
    y="feature",
    orientation="h",
    color="spearman_corr",
    color_continuous_scale="RdBu",
    title="Pre-Tournament Feature Correlation with World Cup Finish Score",
    labels={"spearman_corr": "Spearman correlation", "feature": "Feature"},
)
fig_pre.update_layout(yaxis=dict(categoryorder="total ascending"), height=500, width=950)
fig_pre.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "pre_tournament_feature_correlation",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})

fig_in = px.bar(
    in_tournament_corr,
    x="spearman_corr",
    y="feature",
    orientation="h",
    color="spearman_corr",
    color_continuous_scale="RdBu",
    title="In-Tournament Stat Correlation with World Cup Finish Score",
    labels={"spearman_corr": "Spearman correlation", "feature": "Feature"},
)
fig_in.update_layout(yaxis=dict(categoryorder="total ascending"), height=500, width=950)
fig_in.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "in_tournament_stat_correlation",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})

heatmap_columns = ["finish_score"] + pre_tournament_features 
heatmap_corr = outcome_df[heatmap_columns].corr(method="spearman")
fig_heatmap = px.imshow(
    heatmap_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Spearman Correlation Heatmap: Outcome, Predictors",
)
fig_heatmap.update_layout(height=1050, width=1150)
fig_heatmap.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "outcome_predictors_correlation_heatmap",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})

heatmap_columns = ["finish_score"] + in_tournament_features
heatmap_corr = outcome_df[heatmap_columns].corr(method="spearman")
fig_heatmap = px.imshow(
    heatmap_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Spearman Correlation Heatmap: Outcome, Tournament Stats",
)
fig_heatmap.update_layout(height=900, width=1000)
fig_heatmap.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "outcome_tournament_stats_correlation_heatmap",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})

strongest_pre_features = pre_tournament_corr.head(3)["feature"].tolist()
for feature in strongest_pre_features:
    scatter_df = outcome_df[["edition", "country", "placement", "finish_score", feature]].dropna()
    fig_scatter = px.scatter(
        scatter_df,
        x=feature,
        y="finish_score",
        color="placement",
        hover_data=["country", "edition"],
        trendline="ols",
        trendline_scope="overall",
        title=f"{feature} vs World Cup Finish Score",
        labels={feature: feature.replace("_", " ").title(), "finish_score": "Finish Score"},
    )
    fig_scatter.update_layout(height=550, width=900)
    fig_scatter.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "feature_finish_score_scatter",
            "height": 800,
            "width": 1200,
            "scale": 3,
        }
    })

# Show the analysis dataframe so it can be reused in later cells.
display(outcome_df.head(20))


Analysis rows: 489
Editions: 1930-2022 (22 tournaments)

Pre-tournament predictors vs finish_score


,feature,n,pearson_corr,spearman_corr,abs_spearman_corr
0,start_elo,489,0.483007,0.491823,0.491823
1,prior_best_finish_score,405,0.359745,0.384366,0.384366
2,prior_avg_goal_diff_per_match,405,0.320866,0.351370,0.351370
3,prior_avg_goals_per_match,405,0.294832,0.349616,0.349616
4,weighted_form_l10_goal_difference_per_match,487,0.349838,0.347870,0.347870
5,prior_world_cup_participations,489,0.374668,0.338899,0.338899
6,form_l10_goal_difference,487,0.337740,0.330836,0.330836
7,form_l10_goal_difference_per_match,487,0.331910,0.328792,0.328792
8,prior_avg_finish_score,405,0.298615,0.326071,0.326071
9,weighted_form_l10_result_score,487,0.313861,0.313007,0.313007



In-tournament explanatory stats vs finish_score


,feature,n,pearson_corr,spearman_corr,abs_spearman_corr
0,matches_played,489,0.861094,0.897085,0.897085
1,gs,489,0.782187,0.809571,0.809571
2,goal_difference_per_match,489,0.631593,0.722577,0.722577
3,goal_difference,489,0.705068,0.708284,0.708284
4,finish_elo,489,0.669942,0.678436,0.678436
5,elo_change,489,0.596718,0.582599,0.582599
6,goals_per_match,489,0.524178,0.574395,0.574395
7,goals_against_per_match,489,-0.412742,-0.447514,0.447514
8,ga,489,0.074316,0.077908,0.077908


,tournament_id,year,country,team_id,team_code,confederation,placement,position,matches_played,gs,ga,start_elo,finish_elo,elo_change,next_edition,next_placement,next_position,edition,edition_team_count,edition_finish_scale,finish_score,placement_score,goal_difference,goals_per_match,goals_against_per_match,goal_difference_per_match,is_host,prior_world_cup_participations,previous_finish_score,previous_position,prior_avg_finish_score,prior_best_finish_score,prior_avg_goals_per_match,prior_avg_goal_diff_per_match,form_l10_matches,form_l10_win_pct,form_l10_goals_for,form_l10_goals_against,form_l10_goal_difference,form_l10_goals_per_match,form_l10_goals_against_per_match,form_l10_goal_difference_per_match,form_l10_elo_change,form_l10_elo_change_per_match,weighted_form_l10_result_score,weighted_form_l10_win_pct,weighted_form_l10_goals_for_per_match,weighted_form_l10_goals_against_per_match,weighted_form_l10_goal_difference_per_match,weighted_form_l10_elo_change_per_match,form_l10_first_match_date,form_l10_last_match_date,edition_start_date
0,WC-1982,1982,Algeria,DZA,DZA,CAF,Group Stage,24,3,5,5,1696,1713,17,1986.0,Group Stage,24.0,1982,24,24,0.000000,1,0,1.666667,1.666667,0.000000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,10.0,0.6,13.0,8.0,5.0,1.3,0.8,0.5,19.0,1.900000,0.590909,0.454545,1.200000,0.963636,0.236364,-4.454545,1981-10-10,1982-04-28,1982-06-13
1,WC-1986,1986,Algeria,DZA,DZA,CAF,Group Stage,24,3,1,5,1618,1609,-9,2010.0,Group Stage,32.0,1986,24,24,0.000000,1,-4,0.333333,1.666667,-1.333333,0,1,0.000000,24.0,0.000000,0.000000,1.666667,0.000000,10.0,0.1,9.0,14.0,-5.0,0.9,1.4,-0.5,-69.0,-7.666667,0.272727,0.090909,0.927273,1.418182,-0.490909,-7.644444,1985-12-11,1986-05-11,1986-05-31
2,WC-2010,2010,Algeria,DZA,DZA,CAF,Group Stage,32,3,0,2,1539,1535,-4,2014.0,Round of 16,16.0,2010,32,32,0.000000,1,-2,0.000000,0.666667,-0.666667,0,2,0.000000,24.0,0.000000,0.000000,1.000000,-0.666667,10.0,0.4,6.0,16.0,-10.0,0.6,1.6,-1.0,-11.0,-1.100000,0.381818,0.345455,0.527273,1.781818,-1.254545,-1.709091,2009-11-18,2010-06-05,2010-06-11
3,WC-2014,2014,Algeria,DZA,DZA,CAF,Round of 16,16,4,7,7,1625,1673,48,NaN,NaN,NaN,2014,32,32,0.516129,2,0,1.750000,1.750000,0.000000,0,3,0.000000,32.0,0.000000,0.000000,0.666667,-0.666667,10.0,0.8,19.0,8.0,11.0,1.9,0.8,1.1,102.0,10.200000,0.854545,0.818182,1.927273,0.854545,1.072727,9.818182,2013-06-02,2014-06-04,2014-06-12
4,WC-2006,2006,Angola,AGO,AGO,CAF,Group Stage,32,3,1,2,1565,1595,30,NaN,NaN,NaN,2006,32,32,0.000000,1,-1,0.333333,0.666667,-0.333333,0,0,NaN,NaN,NaN,NaN,NaN,NaN,10.0,0.3,16.0,16.0,0.0,1.6,1.6,0.0,20.0,2.000000,0.418182,0.363636,1.836364,1.690909,0.145455,2.672727,2005-11-16,2006-06-02,2006-06-09
5,WC-1930,1930,Argentina,ARG,ARG,CONMEBOL,Runner-up,2,5,18,9,2073,2059,-14,1934.0,Round of 16,16.0,1930,13,13,0.916667,6,9,3.600000,1.800000,1.800000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,10.0,0.5,17.0,7.0,10.0,1.7,0.7,1.0,43.0,4.300000,0.709091,0.509091,1.836364,0.654545,1.181818,3.272727,1928-08-30,1930-05-25,1930-07-13
6,WC-1934,1934,Argentina,ARG,ARG,CONMEBOL,Round of 16,16,1,2,3,2062,2010,-52,1958.0,Group Stage,16.0,1934,16,16,0.000000,2,-1,2.000000,3.000000,-1.000000,0,1,0.916667,2.0,0.916667,0.916667,3.600000,1.800000,10.0,0.6,17.0,8.0,9.0,1.7,0.8,0.9,3.0,0.300000,0.663636,0.600000,1.690909,0.818182,0.872727,1.581818,1930-08-03,1933-12-14,1934-05-27
7,WC-1958,1958,Argentina,ARG,ARG,CONMEBOL,Group Stage,16,3,5,10,1984,1922,-62,1962.0,Group Stage,16.0,1958,16,16,0.000000,1,-5,1.666667,3.333333,-1.666667,0,2,0.000000,16.0,0.458333,0.916667,2.800000,0.400000,10.0,0.6,16.0,7.0,9.0,1.6,0.7,0.9,-64.0,-6.400000,0.636364,0.636364,1.672727,0.472727,1.200000,-3.945455,1957-07-07,1958-04-30,1958-06-08
8,WC-1962,1962,Argentina,ARG,ARG,CONMEBOL,Group Stage,16,3,2,3,1980,1941,-39,1966.0,Quarter-final,8.0,1962,16,16,0.000000,1,-1,0.666667,1.000000,-0.333333,0,3,0.000000,16.0,0.305556,0.916667,2.422222,-0.288889,10.0,0.3,14.0,13.0,1.0,1.4,1.3,0.1,-36.0,-3.600000,0.536364,0.345455,1.545455,1.254545,0.290909,-2.400000,1961-05-17,1962-03-

### Last-k World Cup History

Repeat the correlation workflow using a configurable recent-history window. The default setting looks back across each team's previous five World Cup appearances.

In [139]:
# Last-k World Cup history correlation analysis

WORLD_CUP_LOOKBACK = 5

# Reuse the previous analysis dataframe when available; rebuild it when this cell is run by itself.
if "outcome_df" not in globals():
    placement_score_map = {
        "Winner": 7,
        "Runner-up": 6,
        "Third Place": 5,
        "Fourth Place": 4,
        "Semi-final": 4,
        "Quarter-final": 3,
        "Round of 16": 2,
        "Group Stage": 1,
    }
    placement_files = sorted(Path("data/processed/world_cup").glob("[0-9][0-9][0-9][0-9]/placement.csv"))
    placement_frames = []
    for placement_file in placement_files:
        edition = int(placement_file.parent.name)
        edition_df = pd.read_csv(placement_file)
        edition_df["edition"] = edition
        placement_frames.append(edition_df)
    outcome_df = pd.concat(placement_frames, ignore_index=True)
    outcome_df = outcome_df.sort_values(["country", "edition"]).reset_index(drop=True)
    outcome_df["edition_team_count"] = outcome_df.groupby("edition")["country"].transform("count")
    outcome_df["position"] = pd.to_numeric(outcome_df["position"], errors="coerce")
    outcome_df["edition_finish_scale"] = outcome_df.groupby("edition")["position"].transform("max")
    outcome_df["edition_finish_scale"] = outcome_df[["edition_team_count", "edition_finish_scale"]].max(axis=1)
    outcome_df["finish_score"] = 1 - (
        (outcome_df["position"] - 1) / (outcome_df["edition_finish_scale"] - 1)
    )
    outcome_df["placement_score"] = outcome_df["placement"].map(placement_score_map)
    for column in ["matches_played", "gs", "ga", "start_elo", "finish_elo", "elo_change"]:
        outcome_df[column] = pd.to_numeric(outcome_df[column], errors="coerce")
    outcome_df["goal_difference"] = outcome_df["gs"] - outcome_df["ga"]
    outcome_df["goals_per_match"] = outcome_df["gs"] / outcome_df["matches_played"]
    outcome_df["goals_against_per_match"] = outcome_df["ga"] / outcome_df["matches_played"]
    outcome_df["goal_difference_per_match"] = outcome_df["goal_difference"] / outcome_df["matches_played"]

wc_history_base = outcome_df.copy()
for column in [
    "position",
    "finish_score",
    "matches_played",
    "gs",
    "ga",
    "goal_difference",
    "goals_per_match",
    "goals_against_per_match",
    "goal_difference_per_match",
    "elo_change",
]:
    wc_history_base[column] = pd.to_numeric(wc_history_base[column], errors="coerce")

all_editions = sorted(wc_history_base["edition"].dropna().astype(int).unique().tolist())
team_edition_lookup = {
    (int(row.edition), str(row.country)): row
    for row in wc_history_base.itertuples(index=False)
}

def safe_mean(values):
    values = pd.to_numeric(pd.Series(values), errors="coerce")
    valid = values.dropna()
    return float(valid.mean()) if not valid.empty else np.nan

def safe_sum(values):
    values = pd.to_numeric(pd.Series(values), errors="coerce")
    valid = values.dropna()
    return float(valid.sum()) if not valid.empty else np.nan

def weighted_mean(values, weights):
    values = pd.to_numeric(pd.Series(values), errors="coerce")
    weights = pd.Series(weights, index=values.index, dtype=float)
    valid = values.notna()
    if not valid.any():
        return np.nan
    return float(np.average(values[valid], weights=weights[valid]))

wc_l5_rows = []
prior_edition_audit_rows = []
for row in wc_history_base[["edition", "country"]].itertuples(index=False):
    edition = int(row.edition)
    country = str(row.country)
    prior_editions = [prior_edition for prior_edition in all_editions if prior_edition < edition][-WORLD_CUP_LOOKBACK:]
    weights = pd.Series(np.arange(1, len(prior_editions) + 1, dtype=float))

    appeared_flags = []
    finish_scores_with_dnq = []
    appearance_positions = []
    appearance_goals_for = []
    appearance_goals_against = []
    appearance_goal_difference = []
    appearance_goals_per_match = []
    appearance_goals_against_per_match = []
    appearance_goal_difference_per_match = []
    appearance_elo_change = []

    for prior_edition in prior_editions:
        prior_row = team_edition_lookup.get((prior_edition, country))
        appeared = prior_row is not None
        appeared_flags.append(1.0 if appeared else 0.0)
        finish_scores_with_dnq.append(float(getattr(prior_row, "finish_score")) if appeared else 0.0)
        prior_edition_audit_rows.append(
            {
                "edition": edition,
                "country": country,
                "prior_edition": int(prior_edition),
                "appeared": int(appeared),
            }
        )
        if appeared:
            appearance_positions.append(getattr(prior_row, "position"))
            appearance_goals_for.append(getattr(prior_row, "gs"))
            appearance_goals_against.append(getattr(prior_row, "ga"))
            appearance_goal_difference.append(getattr(prior_row, "goal_difference"))
            appearance_goals_per_match.append(getattr(prior_row, "goals_per_match"))
            appearance_goals_against_per_match.append(getattr(prior_row, "goals_against_per_match"))
            appearance_goal_difference_per_match.append(getattr(prior_row, "goal_difference_per_match"))
            appearance_elo_change.append(getattr(prior_row, "elo_change"))

    appearances = int(sum(appeared_flags))
    prior_count = len(prior_editions)
    total_weight = float(weights.sum()) if prior_count else np.nan
    weighted_appearance_rate = (
        float(np.dot(appeared_flags, weights) / total_weight) if prior_count else np.nan
    )
    weighted_finish_score = (
        float(np.dot(finish_scores_with_dnq, weights) / total_weight) if prior_count else np.nan
    )

    appeared_weights = [weight for flag, weight in zip(appeared_flags, weights, strict=False) if flag]

    wc_l5_rows.append(
        {
            "edition": edition,
            "country": country,
            "wc_l5_prior_edition_count": int(prior_count),
            "wc_l5_prior_editions": ",".join(str(prior_edition) for prior_edition in prior_editions),
            "wc_l5_appearances": appearances,
            "wc_l5_appearance_rate": float(appearances / prior_count) if prior_count else 0.0,
            "wc_l5_avg_finish_score": float(np.mean(finish_scores_with_dnq)) if prior_count else np.nan,
            "wc_l5_best_finish_score": float(np.max(finish_scores_with_dnq)) if prior_count else np.nan,
            "wc_l5_avg_position": safe_mean(appearance_positions),
            "wc_l5_goals_for": safe_sum(appearance_goals_for),
            "wc_l5_goals_against": safe_sum(appearance_goals_against),
            "wc_l5_goal_difference": safe_sum(appearance_goal_difference),
            "wc_l5_goals_per_match": safe_mean(appearance_goals_per_match),
            "wc_l5_goals_against_per_match": safe_mean(appearance_goals_against_per_match),
            "wc_l5_goal_difference_per_match": safe_mean(appearance_goal_difference_per_match),
            "wc_l5_elo_change": safe_sum(appearance_elo_change),
            "wc_l5_elo_change_per_appearance": safe_mean(appearance_elo_change),
            "weighted_wc_l5_appearance_rate": weighted_appearance_rate,
            "weighted_wc_l5_finish_score": weighted_finish_score,
            "weighted_wc_l5_position": weighted_mean(appearance_positions, appeared_weights) if appearance_positions else np.nan,
            "weighted_wc_l5_goals_per_match": weighted_mean(appearance_goals_per_match, appeared_weights) if appearance_goals_per_match else np.nan,
            "weighted_wc_l5_goals_against_per_match": weighted_mean(appearance_goals_against_per_match, appeared_weights) if appearance_goals_against_per_match else np.nan,
            "weighted_wc_l5_goal_difference_per_match": weighted_mean(appearance_goal_difference_per_match, appeared_weights) if appearance_goal_difference_per_match else np.nan,
            "weighted_wc_l5_elo_change_per_appearance": weighted_mean(appearance_elo_change, appeared_weights) if appearance_elo_change else np.nan,
        }
    )

wc_l5_features = [
    "wc_l5_appearances",
    "wc_l5_appearance_rate",
    "wc_l5_avg_finish_score",
    "wc_l5_best_finish_score",
    "wc_l5_avg_position",
    "wc_l5_goals_for",
    "wc_l5_goals_against",
    "wc_l5_goal_difference",
    "wc_l5_goals_per_match",
    "wc_l5_goals_against_per_match",
    "wc_l5_goal_difference_per_match",
    "wc_l5_elo_change",
    "wc_l5_elo_change_per_appearance",
    "weighted_wc_l5_appearance_rate",
    "weighted_wc_l5_finish_score",
    "weighted_wc_l5_position",
    "weighted_wc_l5_goals_per_match",
    "weighted_wc_l5_goals_against_per_match",
    "weighted_wc_l5_goal_difference_per_match",
    "weighted_wc_l5_elo_change_per_appearance",
]

wc_l5_feature_df = pd.DataFrame(wc_l5_rows)
wc_l5_analysis_df = wc_history_base.merge(wc_l5_feature_df, on=["edition", "country"], how="left")
wc_l5_prior_edition_audit_df = pd.DataFrame(prior_edition_audit_rows)

# Mirror the first correlation-analysis cell's baseline predictors and explanatory stats.
if "is_host" not in wc_l5_analysis_df.columns:
    host_pairs = set(
        pd.read_csv("data/processed/world_cup/all_editions/results.csv")[["edition", "country"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )
    wc_l5_analysis_df["is_host"] = [
        int((int(edition), country) in host_pairs)
        for edition, country in zip(wc_l5_analysis_df["edition"], wc_l5_analysis_df["country"])
    ]

team_history = wc_l5_analysis_df.sort_values(["country", "edition"]).groupby("country", group_keys=False)
if "prior_world_cup_participations" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["prior_world_cup_participations"] = team_history.cumcount()
if "previous_finish_score" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["previous_finish_score"] = team_history["finish_score"].shift(1)
if "previous_position" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["previous_position"] = team_history["position"].shift(1)
if "prior_avg_finish_score" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["prior_avg_finish_score"] = team_history["finish_score"].transform(
        lambda series: series.shift(1).expanding().mean()
    )
if "prior_best_finish_score" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["prior_best_finish_score"] = team_history["finish_score"].transform(
        lambda series: series.shift(1).expanding().max()
    )
if "prior_avg_goals_per_match" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["prior_avg_goals_per_match"] = team_history["goals_per_match"].transform(
        lambda series: series.shift(1).expanding().mean()
    )
if "prior_avg_goal_diff_per_match" not in wc_l5_analysis_df.columns:
    wc_l5_analysis_df["prior_avg_goal_diff_per_match"] = team_history["goal_difference_per_match"].transform(
        lambda series: series.shift(1).expanding().mean()
    )

baseline_pre_tournament_features = [
    "start_elo",
    "is_host",
    "prior_world_cup_participations",
    "prior_avg_finish_score",
    "prior_best_finish_score",
    "prior_avg_goals_per_match",
    "prior_avg_goal_diff_per_match",
    "previous_finish_score",
    "previous_position",
]

wc_l5_pre_tournament_features = baseline_pre_tournament_features + wc_l5_features

in_tournament_features = [
    "matches_played",
    "gs",
    "ga",
    "goal_difference",
    "goals_per_match",
    "goals_against_per_match",
    "goal_difference_per_match",
    "finish_elo",
    "elo_change",
]

def build_correlation_table(df, feature_columns, target="finish_score"):
    rows = []
    for feature in feature_columns:
        analysis_df = df[[feature, target]].dropna()
        rows.append(
            {
                "feature": feature,
                "n": len(analysis_df),
                "pearson_corr": analysis_df[feature].corr(analysis_df[target], method="pearson"),
                "spearman_corr": analysis_df[feature].corr(analysis_df[target], method="spearman"),
            }
        )
    corr_df = pd.DataFrame(rows)
    corr_df["abs_spearman_corr"] = corr_df["spearman_corr"].abs()
    return corr_df.sort_values("abs_spearman_corr", ascending=False).reset_index(drop=True)

wc_l5_corr = build_correlation_table(wc_l5_analysis_df, wc_l5_pre_tournament_features)
in_tournament_corr = build_correlation_table(wc_l5_analysis_df, in_tournament_features)

# Sanity checks for leakage and feature integrity.
assert not wc_l5_analysis_df.duplicated(["edition", "country"]).any(), "Expected one row per edition + country."
assert wc_l5_analysis_df["wc_l5_prior_edition_count"].le(WORLD_CUP_LOOKBACK).all(), "Prior edition window is too large."
if not wc_l5_prior_edition_audit_df.empty:
    assert (wc_l5_prior_edition_audit_df["prior_edition"] < wc_l5_prior_edition_audit_df["edition"]).all(), "Prior editions must be before the current edition."
assert wc_l5_analysis_df.loc[wc_l5_analysis_df["edition"].eq(1930), "wc_l5_appearances"].eq(0).all(), "First-edition rows must have no prior appearances."
for edition, country in [(2022, "Brazil"), (2018, "Germany")]:
    audit = wc_l5_prior_edition_audit_df[
        wc_l5_prior_edition_audit_df["edition"].eq(edition)
        & wc_l5_prior_edition_audit_df["country"].eq(country)
    ]
    assert not audit.empty, f"Missing prior-edition audit rows for {country} {edition}."
    assert audit["prior_edition"].lt(edition).all(), f"Current edition leaked into {country} {edition}."
weighted_diff_check = wc_l5_analysis_df[
    wc_l5_analysis_df["wc_l5_prior_edition_count"].ge(2)
    & wc_l5_analysis_df["wc_l5_avg_finish_score"].notna()
    & wc_l5_analysis_df["weighted_wc_l5_finish_score"].notna()
    & ~np.isclose(wc_l5_analysis_df["wc_l5_avg_finish_score"], wc_l5_analysis_df["weighted_wc_l5_finish_score"])
]
assert not weighted_diff_check.empty, "Expected weighted and unweighted last-5 finish scores to differ somewhere."
excluded_correlation_features = {"edition", "country", "placement"}
assert excluded_correlation_features.isdisjoint(wc_l5_pre_tournament_features + in_tournament_features), "Correlation features include identifiers or categorical placement."

print(f"Last-{WORLD_CUP_LOOKBACK} World Cup analysis rows: {len(wc_l5_analysis_df):,}")
print("\nLast-5 World Cup history features vs finish_score")
display(wc_l5_corr)

fig_wc_l5 = px.bar(
    wc_l5_corr,
    x="spearman_corr",
    y="feature",
    orientation="h",
    color="spearman_corr",
    color_continuous_scale="RdBu",
    title=f"Pre-Tournament Predictors + Last-{WORLD_CUP_LOOKBACK} World Cup History Correlation with Finish Score",
    labels={"spearman_corr": "Spearman correlation", "feature": "Feature"},
)
fig_wc_l5.update_layout(yaxis=dict(categoryorder="total ascending"), height=650, width=1050)
fig_wc_l5.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "world_cup_history_correlation",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})

heatmap_columns = ["finish_score"] + wc_l5_pre_tournament_features
heatmap_corr = wc_l5_analysis_df[heatmap_columns].corr(method="spearman")
fig_heatmap = px.imshow(
    heatmap_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title=f"Spearman Correlation Heatmap: Outcome, Baseline Predictors, and Last-{WORLD_CUP_LOOKBACK} World Cup History",
)
fig_heatmap.update_layout(height=1100, width=1150)
fig_heatmap.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "baseline_history_correlation_heatmap",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})

fig_in = px.bar(
    in_tournament_corr,
    x="spearman_corr",
    y="feature",
    orientation="h",
    color="spearman_corr",
    color_continuous_scale="RdBu",
    title="In-Tournament Stat Correlation with World Cup Finish Score",
    labels={"spearman_corr": "Spearman correlation", "feature": "Feature"},
)
fig_in.update_layout(yaxis=dict(categoryorder="total ascending"), height=500, width=950)
fig_in.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "in_tournament_stat_correlation_with_history",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})

in_heatmap_columns = ["finish_score"] + in_tournament_features
in_heatmap_corr = wc_l5_analysis_df[in_heatmap_columns].corr(method="spearman")
fig_in_heatmap = px.imshow(
    in_heatmap_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Spearman Correlation Heatmap: Outcome and Tournament Stats",
)
fig_in_heatmap.update_layout(height=900, width=1000)
fig_in_heatmap.show(config={
    "toImageButtonOptions": {
        "format": "png",
        "filename": "outcome_tournament_stats_heatmap_with_history",
        "height": 800,
        "width": 1200,
        "scale": 3,
    }
})

strongest_wc_l5_features = wc_l5_corr.head(3)["feature"].tolist()
for feature in strongest_wc_l5_features:
    scatter_df = wc_l5_analysis_df[["edition", "country", "placement", "finish_score", feature]].dropna()
    fig_scatter = px.scatter(
        scatter_df,
        x=feature,
        y="finish_score",
        color="placement",
        hover_data=["country", "edition"],
        trendline="ols",
        trendline_scope="overall",
        title=f"{feature} vs World Cup Finish Score",
        labels={feature: feature.replace("_", " ").title(), "finish_score": "Finish Score"},
    )
    fig_scatter.update_layout(height=550, width=900)
    fig_scatter.show(config={
        "toImageButtonOptions": {
            "format": "png",
            "filename": "feature_finish_score_scatter_with_history",
            "height": 800,
            "width": 1200,
            "scale": 3,
        }
    })

display(wc_l5_analysis_df.head(20))


Last-5 World Cup analysis rows: 489

Last-5 World Cup history features vs finish_score


,feature,n,pearson_corr,spearman_corr,abs_spearman_corr
0,start_elo,489,0.483007,0.491823,0.491823
1,wc_l5_goal_difference,382,0.391735,0.424054,0.424054
2,wc_l5_goals_for,382,0.394838,0.396696,0.396696
3,weighted_wc_l5_finish_score,476,0.393595,0.387492,0.387492
4,wc_l5_avg_finish_score,476,0.389561,0.384460,0.384460
5,prior_best_finish_score,405,0.359745,0.384366,0.384366
6,weighted_wc_l5_goal_difference_per_match,382,0.365913,0.383999,0.383999
7,wc_l5_goal_difference_per_match,382,0.366303,0.382578,0.382578
8,prior_avg_goal_diff_per_match,405,0.320866,0.351370,0.351370
9,prior_avg_goals_per_match,405,0.294832,0.349616,0.349616


,tournament_id,year,country,team_id,team_code,confederation,placement,position,matches_played,gs,ga,start_elo,finish_elo,elo_change,next_edition,next_placement,next_position,edition,edition_team_count,edition_finish_scale,finish_score,placement_score,goal_difference,goals_per_match,goals_against_per_match,goal_difference_per_match,is_host,prior_world_cup_participations,previous_finish_score,previous_position,prior_avg_finish_score,prior_best_finish_score,prior_avg_goals_per_match,prior_avg_goal_diff_per_match,form_l10_matches,form_l10_win_pct,form_l10_goals_for,form_l10_goals_against,form_l10_goal_difference,form_l10_goals_per_match,form_l10_goals_against_per_match,form_l10_goal_difference_per_match,form_l10_elo_change,form_l10_elo_change_per_match,weighted_form_l10_result_score,weighted_form_l10_win_pct,weighted_form_l10_goals_for_per_match,weighted_form_l10_goals_against_per_match,weighted_form_l10_goal_difference_per_match,weighted_form_l10_elo_change_per_match,form_l10_first_match_date,form_l10_last_match_date,edition_start_date,wc_l5_prior_edition_count,wc_l5_prior_editions,wc_l5_appearances,wc_l5_appearance_rate,wc_l5_avg_finish_score,wc_l5_best_finish_score,wc_l5_avg_position,wc_l5_goals_for,wc_l5_goals_against,wc_l5_goal_difference,wc_l5_goals_per_match,wc_l5_goals_against_per_match,wc_l5_goal_difference_per_match,wc_l5_elo_change,wc_l5_elo_change_per_appearance,weighted_wc_l5_appearance_rate,weighted_wc_l5_finish_score,weighted_wc_l5_position,weighted_wc_l5_goals_per_match,weighted_wc_l5_goals_against_per_match,weighted_wc_l5_goal_difference_per_match,weighted_wc_l5_elo_change_per_appearance
0,WC-1982,1982,Algeria,DZA,DZA,CAF,Group Stage,24,3,5,5,1696,1713,17,1986.0,Group Stage,24.0,1982,24,24,0.000000,1,0,1.666667,1.666667,0.000000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,10.0,0.6,13.0,8.0,5.0,1.3,0.8,0.5,19.0,1.900000,0.590909,0.454545,1.200000,0.963636,0.236364,-4.454545,1981-10-10,1982-04-28,1982-06-13,5,"1962,1966,1970,1974,1978",0,0.0,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
1,WC-1986,1986,Algeria,DZA,DZA,CAF,Group Stage,24,3,1,5,1618,1609,-9,2010.0,Group Stage,32.0,1986,24,24,0.000000,1,-4,0.333333,1.666667,-1.333333,0,1,0.000000,24.0,0.000000,0.000000,1.666667,0.000000,10.0,0.1,9.0,14.0,-5.0,0.9,1.4,-0.5,-69.0,-7.666667,0.272727,0.090909,0.927273,1.418182,-0.490909,-7.644444,1985-12-11,1986-05-11,1986-05-31,5,"1966,1970,1974,1978,1982",1,0.2,0.000000,0.000000,24.000000,5.0,5.0,0.0,1.666667,1.666667,0.000000,17.0,17.000000,0.333333,0.000000,24.000000,1.666667,1.666667,0.000000,17.000000
2,WC-2010,2010,Algeria,DZA,DZA,CAF,Group Stage,32,3,0,2,1539,1535,-4,2014.0,Round of 16,16.0,2010,32,32,0.000000,1,-2,0.000000,0.666667,-0.666667,0,2,0.000000,24.0,0.000000,0.000000,1.000000,-0.666667,10.0,0.4,6.0,16.0,-10.0,0.6,1.6,-1.0,-11.0,-1.100000,0.381818,0.345455,0.527273,1.781818,-1.254545,-1.709091,2009-11-18,2010-06-05,2010-06-11,5,"1990,1994,1998,2002,2006",0,0.0,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
3,WC-2014,2014,Algeria,DZA,DZA,CAF,Round of 16,16,4,7,7,1625,1673,48,NaN,NaN,NaN,2014,32,32,0.516129,2,0,1.750000,1.750000,0.000000,0,3,0.000000,32.0,0.000000,0.000000,0.666667,-0.666667,10.0,0.8,19.0,8.0,11.0,1.9,0.8,1.1,102.0,10.200000,0.854545,0.818182,1.927273,0.854545,1.072727,9.818182,2013-06-02,2014-06-04,2014-06-12,5,"1994,1998,2002,2006,2010",1,0.2,0.000000,0.000000,32.000000,0.0,2.0,-2.0,0.000000,0.666667,-0.666667,-4.0,-4.000000,0.333333,0.000000,32.000000,0.000000,0.666667,-0.666667,-4.000000
4,WC-2006,2006,Angola,AGO,AGO,CAF,Group Stage,32,3,1,2,1565,1595,30,NaN,NaN,NaN,2006,32,32,0.000000,1,-1,0.333333,0.666667,-0.333333,0,0,NaN,NaN,NaN,NaN,NaN,NaN,10.0,0.3,16.0,16.0,0.0,1.6,1.6,0.0,20.0,2.000000,0.418182,0.363636,1.836364,1.690909,0.145455,2.672727,2005-11-16,2006-06-02,2006-06-09,5,"1986,1990,1994,1998,2002",0,0.0,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
5,WC-1930,